# Домашняя 1 · Метрики ранжирования и вопрос «а точно лучше?»

**Неделя 4 · занятие 1.** Опора — L5 «Метрики ранжирования»: Recall@k, Precision@k, MRR, MAP, nDCG.

На прошлом занятии мы построили BM25 и **ни разу** не сказали «лучше» — было нечем. Сегодня
появляется разметка, и слово «лучше» становится законным. Вместе с ним появляется способ
сказать его неправильно, и половина занятия про это.

| # | вопрос занятия | чем отвечаем |
|---|---|---|
| 1 | Откуда берётся разметка и что означает «нерелевантный»? | строим судейский набор и считаем, чего в нём нет |
| 2 | Какая метрика что меряет — и какую ты сломаешь первой? | реализуем пять метрик и ломаем одну намеренно |
| 3 | Система A лучше B или просто повезло? | 15 запросов, разброс, парный тест, доверительный интервал |

**Данные.** `data/l4-*.json` — числа лекции L5, порождённые `_research/gen_l4.py` поверх
BM25-ранжирования из L3. Плюс 20 Newsgroups, если ты хочешь пересчитать всё на своём корпусе.
Лицензии те же, что на прошлом занятии: де-факто свободный корпус, никаких некоммерческих наборов.

**Среда.** Colab T4 через VS Code, но GPU не нужен: сегодня чистая арифметика. Всё идёт на CPU
и заняло бы секунды даже на телефоне.

**Бюджет: ≈120 минут.**

**Артефакт на вынос.** `artifacts/metrics.py` — твои реализации пяти метрик, и `run.json`
с замерами. На неделе 7 они станут мерилом для каскада: без них там нечем будет доказать,
что переранжирование помогло.

**Как запускать.** Сверху вниз. Ноутбук самодостаточен: индекс с прошлого занятия он
подхватит, если найдёт, а если нет — возьмёт готовое ранжирование из `data/l4-metrics.json`
и скажет об этом.

<details><summary>Почему целое занятие на арифметику из пяти формул</summary>

Потому что формулы — это не сложность. Сложность в том, что метрика превращается в решение,
а решение стоит денег.

Каждая из пяти метрик отвечает на **свой** вопрос, и почти все споры про качество поиска —
это спор двух людей, которые считают разные метрики и думают, что говорят об одном. Recall@100
отвечает «попало ли нужное в кандидаты». Precision@3 — «не стыдно ли показать первый экран».
MRR — «быстро ли пользователь дойдёт до первого полезного». nDCG — «насколько порядок близок
к идеальному». MAP — «насколько хорошо система собирает **все** релевантные, а не один».

Выбрать метрику — значит выбрать, какой продукт ты строишь. Поиск по документации и поиск
товаров нельзя мерить одинаково: в первом пользователю нужен один точный ответ, во втором —
десять сравнимых вариантов. Это решение принимается один раз и потом определяет всё, включая
то, какая модель «выиграет» на твоём бенчмарке.

Вторая причина: метрика — это первое, что начинают оптимизировать, и первое, что ломается
от оптимизации. Ей посвящена часть 4 и отдельное задание.
</details>

<details><summary>Как выбирают метрику под продукт — и почему это решение принимают до эксперимента</summary>

Метрика — это формализация того, что ты считаешь успехом. Выбирать её после того, как увидел
результаты, — то же самое, что менять гипотезу под данные.

**Навигационный запрос** («сбербанк вход в личный кабинет»). Есть ровно один правильный ответ,
и он должен быть первым. Метрика: MRR или Precision@1. Recall бессмысленен — второй правильный
ответ не нужен.

**Информационный запрос** («как работает BM25»). Правильных ответов много, пользователь читает
несколько. Метрика: nDCG@10 с градациями, потому что «отличная статья» и «упоминание в списке»
это разные вещи.

**Транзакционный запрос** («купить наушники»). Пользователю нужен выбор — десяток сравнимых
вариантов. Метрика: Precision@10, а лучше — метрика разнообразия рядом с ней, потому что
десять одинаковых товаров это провал при идеальной точности.

**Первая ступень каскада.** Задача — не потерять. Метрика: Recall@100 или Recall@1000.
Precision здесь мерить бессмысленно: переранжирование всё равно перетасует.

**Юридический или медицинский поиск.** Цена пропуска несопоставима с ценой лишнего документа.
Метрика: Recall@k при фиксированном бюджете просмотра, и порог по нему — часть требований,
а не результат оптимизации.

**Правило, которое дороже таблицы.** Метрика фиксируется **до** того, как ты увидел числа,
и записывается вместе с порогом принятия решения: «катим, если nDCG@10 вырос и Precision@1
не упал». Иначе всегда найдётся метрика, по которой твоё изменение выиграло, — при десяти
метриках и пяти конфигурациях у тебя пятьдесят попыток найти победу, и одна из них найдётся
случайно. Об этом же следующий блок про множественные сравнения.
</details>

<details><summary>MAP целиком: почему это площадь под PR-кривой и что это меняет</summary>

Average Precision выглядит как произвольная конструкция: сумма Precision@k по позициям
попаданий, делённая на число релевантных. У неё есть точный геометрический смысл.

**Вывод.** Построим PR-кривую: по горизонтали Recall, по вертикали Precision, точка ставится
после каждого документа выдачи. Recall меняется скачком **только** на релевантных документах,
причём каждый скачок равен `1/R`, где `R` — общее число релевантных. Площадь под ступенчатой
кривой равна сумме по скачкам: `Σ (1/R)·Precision@k_i`, где `k_i` — позиции попаданий. Это
в точности AP. То есть **AP — площадь под PR-кривой**, посчитанная методом прямоугольников
по точкам попаданий.

**Что это меняет практически.** Во-первых, становится понятно, почему AP чувствителен ко всем
релевантным документам, а не к первому: каждый даёт свой прямоугольник. Во-вторых, ясно,
почему AP плохо работает при неизвестном `R`: знаменатель берётся из разметки, а если она
неполна, AP занижен ровно пропорционально. В-третьих, объясняется, почему AP и nDCG иногда
расходятся в оценке двух систем: AP не имеет дисконта позиции как такового — вес позиции
возникает косвенно через Precision@k, и он падает **быстрее** логарифмического:
по одиночному попаданию — как `1/k` против `1/log₂(k+1)`, при `k=10` это 0,100
против 0,289. Поэтому AP строже nDCG наказывает глубокие находки.

**Разница интерполированного и неинтерполированного AP.** В старых работах PR-кривую
сглаживают, беря максимум Precision правее текущей точки Recall (интерполяция по 11 точкам).
Числа получаются выше и не сравнимы с неинтерполированным AP, которым считают сейчас все
современные инструменты. Встретив в статье значение MAP, стоит понимать, какой из двух.

**MAP против nDCG: когда что.** MAP определён только для бинарной релевантности — градации
он не умеет. nDCG умеет градации, но требует выбрать функцию выигрыша и дисконт. Если
разметка бинарная, MAP предпочтительнее: у него меньше произвольных решений внутри.
</details>

## Шаг 0 · Пины и preflight

Сегодня зависимостей минимум — вся тяжёлая работа была на прошлом занятии. Но `preflight()`
на месте: он проверяет доступность чисел лекции, без которых половина сверок бессмысленна.

In [ ]:
# ПИНЫ — точнее, ОТКАЗ от них там, где они ломают Colab.
# Базовый стек образа (numpy, scipy, scikit-learn, matplotlib, torch) собран сам под себя.
# Понижать его нельзя: `pip install numpy==1.26.4` откатывает ОДИН numpy, а scipy и sklearn
# остаются собранными под numpy 2 — и первый же импорт падает с
# «ModuleNotFoundError: No module named 'numpy.char'». Ставим ТОЛЬКО то, чего в образе нет.
import importlib.util as _ilu, subprocess as _sp, sys as _sys

NEEDED = {}
_missing = [pkg for mod, pkg in NEEDED.items() if _ilu.find_spec(mod) is None]
if _missing:
    print("ставлю:", ", ".join(_missing))
    _sp.run([_sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    print("готово · если следующий импорт упадёт — Runtime → Restart session, потом эта ячейка снова")
else:
    print("всё нужное уже в образе Colab — ставить нечего")

import json, math, os, random, statistics
from itertools import product
from pathlib import Path
from IPython.display import HTML, display   # ядро Jupyter/Colab: есть всегда,
                                            # ставить нечего, версия — версия ядра.
# google.colab (drive.mount) — часть рантайма Colab, не пакет: вне Colab его просто нет,
# и ноутбук честно ловит ImportError и пишет артефакты рядом с собой.

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ДАННЫЕ ЛЕКЦИИ. Занятие сверяет свои результаты с числами лекции, а те живут в папке
# data/ курса. В Colab её нет — и раньше ноутбук падал на первой же сверке с
# FileNotFoundError. Файлы крошечные, поэтому вшиты прямо сюда: ниже точная копия нужных
# data/*.json, сжатая zlib и записанная base64. Гейт _research/check_notebooks.py следит,
# чтобы копия совпадала с оригиналом байт в байт, — так что разойтись они не могут.
# Если рядом уже лежит настоящая папка курса (или задан DLS_DATA), берётся ОНА.
import base64 as _b64, os as _os, zlib as _zlib
from pathlib import Path as _Path

_DATA_DIR = _Path(_os.environ.get("DLS_DATA", "./data"))
_EMBEDDED = {
    "l4-goodhart-steps.json": (
        "eNrVl71u2zAQx/c8xUFTgiqyvpUEyNBURdChHdIsRRAEtETbamTRoOi2QRGgD9F36d5H6ZP0SNqyLMmKlSWtBps+8vg7"
        "HsW/ed8PAIy7lCXGGRiXjKUzwgV8ZfyepkC/kfkip/Dnx0+4ePfh9dWn4ynJCojfXIKgfF7ChHEQMwpTMsfxhwu2WOaE"
        "Z+LheJLxUhwB4ynlcBOHZhyZcWDGvhl7ZuyasW3Gzi18KZX/jBW0FHDx3g3qLp7ycpWjrXydWwuuaE6/kCKhcA5lklnl"
        "gmD70AeuO8QZqNmVp3dkwUdcFIY3zgrCH+CdDP8cspSSfAU7JHkOG3/QwVurZRXocG5bge/55jrSlS2MwlClZ8zEDP0X"
        "nKXLJBtj0iaczeXaSqpzZRmmynXJljyhMt13HDsJT2ajKS3uct9aPMDvXzgLSUtIiSCj3Dsez93A+lyyQnatVsCrBCiI"
        "a3+4hIQIOmU8o6XmZIWghZCYKkW6Y73IayZIjv2+Hp4mU/zhWkHohJXlWgaO5hs0AHxXn3IKUtyj1THXhiyVoNgzKgtS"
        "tkakWZmwpQrIsezKnLBC8Gysrcr4aHay3CbL3YuFO+TZpx043dFH9JpEe09i0IkL+lh+k+XsyfI9O+rEyY4+YtAkhk2i"
        "3U30TsLufPZuX9jERfvigtAdjouauGBfHD7DcSdNnL8vzgmCPhx+3laH8e23BZez40nBB16t3mHVCNYWue8obvoc6wOv"
        "RAwdddSGUrzqTMP21tf3pZ60+orqx7x+DOsHBF9g1bzVFkNsCckmgx1i0vVKtvO4U1I6E1ntXBfZbZOjvcnbAjMc7rXh"
        "wQB48Hyy3yb7A8h15RkOD9pwrw13dsHrItSAy66n+WGb7w7h11Spyceup/lRm28P4ddkqsnH52n+SZvvDOHXdKvJx641"
        "f0sC9AXDsbxT53RjWoua3jglYTKDuoGPbuCcKGradxWmUegJV/eyjW0jk3L0SCsheuuBUpZUXgx9kdtXF70dGunu0Et7"
        "h3b+37o45Iz26GJ19RoqjdGLSeMQdeiRxupaNlQdh/wp9Kij/SxptF9YGof8L/VIo/1P62JUBb6ti9UdT13ttgVyrYvS"
        "t6WLsjbt1EU5uq6LcuBGFwsmVHV6IStarJu3Kn9Z9lpwLSt+TlKabqzAJvVKflVX4+Sh65qQ+8faQZWyR5BmkwnlJYxp"
        "QpZYIWcC8KsEe+SMPD13aQKGIi2WcfB48BfmbJ92"
    ),
    "l4-graded.json": (
        "eNrVmctu2zgUhvd5CsKrBGPLIqmL3SKbXiYTYDAdTLsbtAEr0TZbmTQoeZogyHYws+07zDN030fJkwxJWTeLMqw6cBAv"
        "bPmQPOfnT+qzZN2eADC4ikU0eAYGF5LENAaSJvQvwiMKZkKCbEHBrx6Qa84ZnwN6TZarhDrggjCeAncMx/gZSCPmpCui"
        "hqhUKfjCsgXIZqcmdHb/738InKaZFGr8gmXpGbj/+yuY62oAD4FQJWQtRdUIVeNsNsrEikUm7DqAv3p5kRd48ebdL0Ye"
        "vV4JTnnGSALmShY4RR/UJO7/+QrPAOGx6ZQwTok07cN8vEo0vtTZ9DQpiRbOYGjsSMVaRlQ7ciVpqkZFi/Gc8qvEc1Y3"
        "4Ps35RCJUxCTjIwTPPq4RL7zKRVcNxnhKZhJsQTI/e0CRCSjcyFvwE+VI3kd0/VtRBJd6lZFVMzVVcs5m34qCnV0SeSc"
        "cZIkN8UKZUU71u0LNl/U21TTnakjCf9MY9XlT9M7r6TiTMcGr4JNFhVRWnVI0kgthZCZsxDRZ3pTddCpVA9YBswcVMQ1"
        "gbuhtQJuVSgXu5UatVLjXanD/uJxP/Goh3ivn3i/v3i/n3i3h/iglRruSu31Fx/2Ew97iJ90iVfv7/OzTfPqkv+h+r+R"
        "MZXl+eDmQzfbwvoNNr+VOVlMSWJAWGbDjfHQNn7ro8yWE6piQRzN9Q5yJn6wWXZV0MR8Z4JwEeN5zHWCAKHypK8xscqo"
        "PfhZyOU6IdpOQ0kwAoXRm4oTB7po2qwIkYO9ELVK+gEuS66o/F2kLGOCt2HTTY4yErM0EmuuVxo6VVjZoi1udFWza8VU"
        "v5eCZ5J9NNIanRsN1j3XjR+bPjVz7E4tErFFYmiXCJ3JFE3sKj3Hg4G/Syjez0jX8R/HSG9fIz3shocaiabILhI7LvTC"
        "XTr9fX3Ek2D6OFZawNwh0Q+QRSK0SIRdEhs5tlTqtl1Cw729VK/H8XKyt5fQ9w/2sp5j20vVtvULVfVYa4amTW5bf7tA"
        "5U9p/o4IbEfM5/uK/++oXKa1/IWBFoTb1tm+dp1w1z/traUrF89WHVmqY2v10F69ie6GgBp1d2rAhzpQo3J/B7zDHagz"
        "tyGgxsudGvzDHajztL8JgUUAtAqAnQJqpNsSUIJup4bwAUyogbC/CZMHMKGGqG0TCkLljGpQ4u16uX2hqKKvr1caTYPw"
        "+7f8LFP3vOZY7zd1DPWx9rY8ViXA+SbPoLziVBfVr3PpdRKFLa7Bbq5ZDhpzYD+Mul6nWhfqwqeMup5brGt/+T+Gup7V"
        "O1GXNz1R1PUV0I0694lyrq+Abs65VsixknLNW1/W5Jw6wdWr4NyGeYZtfh6HJf/Oi0zVP3P5Hf9ljUW3NQZuA7B0ED8A"
        "ALOD4bcH5vBDYG4/oOV304cD7VjoOi6kjoijI4LniIg5DCapAUnjXzu6YQiuMQTvwZA8SXGtxEVmnlPoWYwgKFQCOE7E"
        "HJ2is3P4HBiY6CcSPBslQj+52DykwEM8hEPoGCCd3J38D4NUz6E="
    ),
    "l4-metrics.json": (
        "eNqdl8Fu4zYQhu95ioEuSQCv1rYs29sih2wXCIpuuoGxt2IR0OJYYk2RKkk5MBYB9iH6DH2F3vso+yQdUrZl77qunBws"
        "kUMNPw6H/zCfLwCiR66z6AeIZkwthcqhRGdEZmGhDbwfweXHAuHB6JW33RldK24vY5hhbdHC++TSwtv7YQqm+fxHMChx"
        "xVSGcAOWlQjD/q93kDGHuTZrYBYcefyjRmoI5VA5uLKZiG3FMryO4a1QjEw7NzF8UNvxV/ez2c1s1oP724eb2wewhX5S"
        "gVS46zjqhfVYXZsM/ZIeDVpkJite56ge5Siu1vDP3+SacQucOfZaJq/m5TCNf7daNaYt/MLo8gBdoG0mCCg/B3I/SeCG"
        "q82Xzq96t5rmA0XDpuFtO+ijdkxS7yj0Li29/kZvAINeeCTNI20eU/r91HxPQUa+G/05/FK/8H3Ru3HU2/YQtO8ymBGK"
        "Ni4udLbEdTvAKlFVGAZxwRVYRGBqDbYUkpZKaWAtyxGshsrRniErw8/XL3+1Tmg95KDftomPOgah/dw7Cpl8B7mL11E6"
        "gtKUALhCAwXtJqf91hwqo3mdOdodKZFDswvBYp3IlvYI5+BbzuEpzsnLgplRWqCxlPqK05PSKoQuY4pxBrp2UGkrnCBD"
        "MxSYdIWu8wK4YQuHvEuEk1PkwzMjPEfnOQRHBnSqQTHL6GjVzknKCeuMVqymHMgMPvnDW1KcSQa6RHh0ijN9YbqSQGnh"
        "QGdOVzVFel3OtQQtOZ3ptQWJLK8pYQyF24ic5ER6YNMlrukp3v6ZcXWFUEuQzOQovaDZWjrABckVBbNCXVF4vRZ5Zc1J"
        "WSt4MpQZ1JJhRzrl8PgU8ehlEW7AVSErBnO0DlaUGxpyr+ZsxYRkc0JX+gkyiitlTEmHtEt4J6dgB2eG14ZaBKRWS3gq"
        "mIMF6QDXYEWuoNQGYU1y4dPBYpdIThu4VmzRa8ut+4VsDWs08IuKN8uKktAablQ6SkNz25r6KeL+xWa5UUXuhKVj/78O"
        "E/o7cDnac0kT7FwuhLFutikps42kNeimZYnKgxar9k0HLZ7lAXqyAaCNCT3DOB0Pxk0pa3r68Xgybnq4sJQEytnvq9K2"
        "GvxnVmy/bULV7r5WdAWZN6E5mjFmb7FHt3TPM7Em/TdHnXvDKf9JF/Ldlp9DPupGPkr6k6POveGU/7QbeTIdvzkfftwN"
        "PknHw6POveGU/0lH+N05OQd+2hF+kB7fVm/4Rih84ZQfDJX73RlorzltOW4LSKt17Z2tvXC0hdEL+G4Wr768VQ59MB8c"
        "3v72Ly/7ZXa/IOxfxPavDPtljjjD66cNz+74p6Nkq0pKu3DZrqgYU60Tbv0qCNP2PwL4+uVPkFovLbDMiRX2wJHFgnr3"
        "0x1c3dGVrWDGXfuZni+eL/4FUCEsPg=="
    ),
    "l4-multiquery.json": (
        "eNqtVc1um0AQvvspRlwaqwS8/Bi3EVJdqVKl1hWhuVVRtMGbmAZYZ1knsaJcK/XaJ+hb9N5HyZN0dsHYCVBZVZFgd4aZ"
        "75sflrkfABhnc54Yr8E4ueWH1ysm1sDuaL7MGJQcZnH8+P1nHAMt5jCbRihMI6CCAU3kimbZGuYs50UpBZVsbsExgRDk"
        "gsFHF97OHB8ELa7S4vJFCYJl7IYWCYMllZKJ4giOHbSmULKEI37FfpvKBerm6cUFE6yQG2s4kHwJi1QClRoVyNBSAdqz"
        "OiRFmzNalMBvmNCivOUaNmWlZZg63ZKvRMJUxmeClYyKZGFfsuIs86zlGn7/UilcCJ5D5h6e545vfS15sQ3eVEGX6wLR"
        "ZZpg9hhbnhZpiVLFcFUi+BfcARBTL261+NUyweeptrwmaHmvlQYSbP0ARma9IeY/afR6atbYApFHVs1v0OVTUX5owkCR"
        "7AgoVrYNsEKrNLXiYfPGcLscXbxavo7fcva7nL2WZ9tx0uXoP3Mk23AHm+dD1QKntwV/qXP7VUcv2i0gmzpuWhB4wd5N"
        "IB1N6ChkZxfG43GwRy07mzBuOQb/qwtNMd6nMhIsScsUfyU7LdiBVCdeYexg4r/g0yo/Z+KZfrmBalWN3S2VsRG9ISGx"
        "8bZGeBnP02nTuj20Tg9tq+RbZjd0bDesDPZg9nqY3V7mwO/k9ULX9kL1eq+Mgx5er5fXD4jXyRyEnh2ElYHx5As4bb6A"
        "d7X1QdUSeFmXUG9UyHqjEIZgg4djQ5+ekdEgfF7lGCTDOcR1Z6xXkwkxm5Ot/t7yhEuaqSSa858LsVM0I68P5thxK0XB"
        "pZ4VcUxUBibEsaM+G3j89kMNH11QOMC5CAwHF46dOB4ewTSqzaeRE+pIK4dpFFbgT1ym0dAyBg+DPwVSqAU="
    ),
    "l4-systems.json": (
        "eNqlV8tu4zYU3ecrLgwUzSC2I8p6WDMI0Lww7WIGLRLAQAdpoEh0TIwsaUQ6j2ayLbouuupfdDX7fEA/Yr6kl5RISrbS"
        "x3gRh7qvw3tJnXv1sAMwuEyLZPASBue3BVRUVIzexBnwey7okn8NJa1GH1a0uof85Pj1N8SB4oZWQHyQUkY57KZU0GrJ"
        "csYFS9AzFwuKqyFwStMD13EDJ3D8V3CmYsIRHEAMYpXTFI7euD7cMrFACV/GWQaiWlGg6TWtca5iTjOWUzh8MYYzdp2z"
        "OUviPKEvoYxZhSHESFAuhjBjWVLcFTlwtKLpqIrz90Ogd3EilGg0z1gp01muRCwYGtZ+ZbbisKRxDimbzyHOU4j8r+D4"
        "uzEcF8tyJRBD7ZAnrLyH2wWtKMQ3Mcviq4y+srgz5Yu5d0BKiKV9xgtgyzKjS5rLiMs4X2G697DL8pSWFH9yAcW8Rnkx"
        "HgzV2fBiVSVUHs9lRTmNq2Sxf03zy8wb416ePtlqrx3UWvHrcCr2od46RpXFVpr8h/owUUb82lYd1iEK3uEjgDP2/Vol"
        "1140meh16JulPwkCs/aIXgZ+YEwCMrXyydQEIXYZuVNr0QroO2ELf2qBJqFd+wSXF60UjlopBKHdiE+IcQsi4hq5E3p2"
        "szY3TMLI/dAz2wrCyG43nJgSTQPSKoBvbQLHYoUTG4eEkSmBE01NGniw8nTuT/B6tnIhrmdiOmTa7G2ED4GupXxwI4OG"
        "iZlsHMch1siz23a8trfnWUUwtd7ERiVOZG1atUA0axN4oclHvmvyXmHKTl1GJTqqRV69RyVqMsZYkyisjzS1MjzMWkat"
        "jIT17U1Y5Ldq5Tiu3XsYura2ikPOz5EJ0PxBmQzk0h275l4PUhmdNBUelHpLRiLeqLfZuKH0UYW/bbjBxp5JI3/sNJ6z"
        "pfFsCSUloShq27FcyVpmzT68MCIGsMU8FnNJxaJIJYmss6GiWPcnUvNDOzd9F9sRJf7EDYPpRn7nDZ80gLqlnMbJAvIi"
        "H/1Mq6JFUIpoP/71+0eQJI1sGPMEGZDl10NggrcZXHGqfEai3cPGwVdLyZJlwZlgN1QF4Ej+I6vL6XXc0aEKy7c72xvO"
        "Rg2zSsp7W+Q/4r4M5UnZ901cKSTGsAmIQn3iVXHLzf2CJu/6rpi7aN4wJY+v+MkzKrlPCahPVgllzrKGe4NG9jj8JzBD"
        "AJtgXVUD5m4Fpt/+HrCOqgGbfDFYh8E20dw+NK8XbfTfUjPstwnWVTVg/lapGd7tQeuoGrTgi1PrkHkPmteDFm6FFjxf"
        "yaCvktNtjs30ph6wjqoBi7a5/qaxbYJ1VfrFdrZDmzyP1pMa2YpHTAPvQYt6To1sQyR2dNhAW1NptMk2aGZg2kTrqjSa"
        "tw0ajjzPoXVVGu3fqET9v/h/M8LauEHvykoGVa0UC1z3TfwE+/zLb2t9UskHptMn7Ns4m89YKhabfb7+XoIFWoxupQk6"
        "i8ukYkJ+opydDvFP9ud0//Ovf+RjrWNcfgeOhBweICuK96sSPybnB8SzXVoco6XqV0RTVC07M19G4vLBGUc4+IHyfYR2"
        "QB2Hp52hUbb1duPnHyrxVnWqaainPk71TKmPC0WnTQXrUDIf/BI+aCLvK3enfpZuGn3Rqp0cr0J/XaHjqjyfPjX+dSQ0"
        "14F6p2KUM/yorG7UIKlnkvbU25p77RXSPq2UMCA8/dlgIvi7JkjjfqHuw87jzt+6FK6U"
    ),
}

_DATA_DIR.mkdir(parents=True, exist_ok=True)
_new = [n for n, b in _EMBEDDED.items() if not (_DATA_DIR / n).exists()]
for _n in _new:
    (_DATA_DIR / _n).write_bytes(_zlib.decompress(_b64.b64decode(_EMBEDDED[_n])))
print(f"данные лекции: {len(_EMBEDDED)} файл(ов) в {_DATA_DIR} · "
      f"распаковано {len(_new)}, остальные уже лежали на месте")


In [ ]:
def preflight():
    problems = []
    for name in ("l4-metrics", "l4-graded", "l4-multiquery", "l4-systems", "l4-goodhart-steps"):
        if not Path(f"{DATA_DIR}/{name}.json").exists():
            problems.append(
                f"нет {DATA_DIR}/{name}.json -- сверка с лекцией невозможна. "
                "Смонтируй Drive или положи папку data рядом с ноутбуком.")
    if not INDEX_PATH.exists():
        problems.append(
            f"нет {INDEX_PATH} (индекс с занятия недели 3). Это НЕ ошибка: ноутбук возьмёт "
            "готовое ранжирование из data/l4-metrics.json. Но пересчитать на своём корпусе "
            "не получится.")
    for p in problems:
        print("!", p)
    print("preflight:", "ЧИСТО" if not problems else f"{len(problems)} замечани(я/й) -- читай выше")
    return not problems

## Шаг 1 · Конфигурация

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SMOKE = os.environ.get("SMOKE", "0") == "1"
N_TRIALS = 200 if SMOKE else 20000     # прогонов Монте-Карло для случайного ранжирования
DATA_DIR = os.environ.get("DLS_DATA", "./data")
DRIVE_DIR = "/content/drive/MyDrive/dls-2026"   # накопительная папка курса (правило 10.4)


def _artifacts_dir():
    """Куда класть индекс, эмбеддинги и прочее, что подхватят СЛЕДУЮЩИЕ занятия.

    На Drive, а не в песочницу: песочница Colab умирает вместе с сессией, и цепочка
    занятий рвётся — четвёртое занятие уже не найдёт индекс третьего и молча соберёт
    уменьшенный свой. На Drive артефакты переживают и перезапуск рантайма, и неделю
    между парами, так что к концу курса собирается одна система, а не семь огрызков.
    """
    if os.environ.get("ARTIFACTS"):            # явное указание сильнее всего
        return Path(os.environ["ARTIFACTS"])
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive", force_remount=False)
        return Path(DRIVE_DIR) / "artifacts"
    except Exception as _e:                    # не Colab либо отказ в доступе — не беда
        print(f"Drive не подключён ({type(_e).__name__}): артефакты лягут рядом с ноутбуком.")
        print("Занятие отработает целиком, но следующее не подхватит их и соберёт своё.")
        return Path("./artifacts")


ARTIFACTS = _artifacts_dir()
INDEX_PATH = ARTIFACTS / "bm25_index.json"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Журнал прогона. Всё, что печатают ячейки ниже, дублируется в файл и в конце занятия
# уезжает архивом по ссылке: тетрадка, сохранённая без выходов, теряет диагностику.
# Две тонкости, обе проверены на живом прогоне:
#   · подменять sys.stdout целиком НЕЛЬЗЯ — ipykernel опознаёт поток по типу, и подмена
#     уносит весь вывод мимо тетрадки: ячейки остаются пустыми, студенту смотреть не на что;
#   · патч .write надо ставить ЗАНОВО перед каждой ячейкой — IPython сам оборачивает write
#     на время ячейки (складывает вывод в историю) и снимает обёртку после, поэтому
#     одноразовый патч молча перестаёт писать со второй ячейки.
import sys as _sys

NB = "hw-ranking-metrics"
RUN_DIR = ARTIFACTS / "runs" / NB
RUN_DIR.mkdir(parents=True, exist_ok=True)
_LOG = open(RUN_DIR / "run_log.txt", "w", encoding="utf-8")


def _arm_log(*_a):
    _st = _sys.stdout
    if getattr(_st.write, "_dls_log", False):
        return
    _orig = _st.write

    def _w(data, *a, **k):
        try:
            _LOG.write(data)
        except Exception:
            pass
        return _orig(data, *a, **k)

    _w._dls_log = True
    _st.write = _w


try:
    from IPython import get_ipython as _gi
    _ip = _gi()
    if _ip is not None and not getattr(_ip, "_dls_log_armed", False):
        _ip.events.register("pre_run_cell", _arm_log)
        _ip._dls_log_armed = True
except Exception:                      # не IPython (прогон файлом) — журнал всё равно пишется
    pass
_arm_log()
print(f"журнал прогона: {RUN_DIR / 'run_log.txt'}")

RUN = {"seed": SEED, "smoke": SMOKE, "n_trials": N_TRIALS}
print(json.dumps(RUN, ensure_ascii=False))
preflight()

**Что видно.** Конфигурация напечатана, и `preflight()` сказал, есть ли у нас индекс с прошлого
занятия. Сравнивать надо не строки вывода между собой, а **факт наличия файла с ожиданием**:
если индекса нет, это не поломка, а разрешённый режим — ноутбук возьмёт готовое ранжирование
из данных лекции. Механизм такой по правилу накопительной системы: артефакты переиспользуются,
но каждый семинар обязан запускаться в одиночку. Чего этот вывод НЕ показывает: корректности
самого индекса, если он есть — мы его сегодня не перепроверяем. Что делать: если стоит
`smoke: true`, помни, что Монте-Карло ниже станет грубее, и разброс оценки вырастет.

---

## Часть 1 · Судейский набор — 20 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 1.1 | Что такое qrels и откуда они берутся? | смотрим на разметку лекции и на её происхождение |
| 1.2 | Что означает «нерелевантный»? | считаем, сколько документов вообще просужено |
| 1.3 | Насколько судьи согласны между собой? | считаем каппу на синтетике — без обучения |

### Шаг 1.1 · Разметка, с которой мы работаем

`qrels` (query relevance judgments) — это тройки «запрос, документ, оценка». Всё оценивание
поиска стоит на них, и потому все системные ошибки оценивания живут тоже в них.

In [ ]:
M = json.load(open(f"{DATA_DIR}/l4-metrics.json", encoding="utf-8"))
RANKED = M["ranked"]
RELS = [d["rel"] for d in RANKED]
N_DOCS, N_REL = M["n"], M["relevantTotal"]

print("запрос:", M["queryIntent"])
print(f"документов: {N_DOCS} · релевантных: {N_REL} · доля: {N_REL / N_DOCS:.0%}")
print(f"{'ранг':>5} {'id':>4} {'rel':>4}  категория")
for d in RANKED:
    print(f"{d['rank']:>5} {d['id']:>4} {d['rel']:>4}  {d['cat']}")

**Что видно.** Ровно половина документов помечена релевантными, и метка совпадает с категорией
20NG: `sci.space` → 1, `rec.sport.hockey` → 0. Сравнивать надо не документы между собой,
а **метку с её происхождением**: это не суждение человека о том, отвечает ли документ на запрос,
а автоматическая подстановка по категории. Механизм понятен и удобен — разметка получается
бесплатно и детерминированно. Чего эта таблица НЕ показывает: того, что документ из `sci.space`
может не иметь никакого отношения к запросу `space`, а документ про хоккей — случайно
упоминать космический шаттл. Что делать: держать в голове, что все числа ниже посчитаны
относительно **прокси**, и переносить их на человеческую разметку нельзя.

<details><summary>Почему полноту в поиске почти невозможно измерить честно</summary>

Recall требует знать знаменатель — **все** релевантные документы коллекции. В нашей игрушке
их четыре, и мы это знаем. В любой реальной задаче это число неизвестно принципиально.

**Почему неизвестно.** Чтобы его узнать, надо просудить весь корпус по каждому запросу.
В части 1 мы посчитали, что для миллиона документов это сотни тысяч человеко-часов. Значит,
знаменатель всегда оценивается, и оценивается снизу: считается число релевантных **среди
просуженных**. Отсюда любая опубликованная полнота — это полнота относительно пула, а не
относительно коллекции, и он систематически завышен.

**Три обходных пути, которыми пользуются.**
Первый — не мерить полноту вовсе, а мерить метрики, не требующие знаменателя: Precision@k,
RR, RBP. Для продукта этого обычно достаточно.
Второй — bpref и infAP: метрики, специально устроенные так, чтобы зависеть только от
просуженных документов и не наказывать за находки вне пула. bpref считает долю пар, где
релевантный документ стоит выше нерелевантного, игнорируя непросуженные вовсе.
Третий — оценка знаменателя выборочным досуживанием: взять случайную выборку непросуженных,
разметить, экстраполировать. Дорого, но даёт интервал вместо точки.

**Где полнота всё-таки честна.** Внутри каскада. Если первая ступень отдаёт сто кандидатов,
а вторая переранжирует, то полнота первой ступени меряется **относительно того, что нашла
вся система целиком**, и знаменатель известен: это объединение выдач всех ступеней. Это
не абсолютная полнота коллекции, но ровно та, которая влияет на решение «увеличить ли
глубину первой ступени». Именно так мы будем его использовать на неделе 7.

**Практическое правило.** Увидев в статье высокий Recall@1000, спроси: относительно чего?
Если относительно пула, собранного из систем того же класса, число говорит «мы не хуже
предшественников», а не «мы находим почти всё».
</details>

<details><summary>Как измеряют величину смещения от пулинга — и почему TREC до сих пор работает</summary>

Мы сказали, что непросуженное считается нерелевантным и что это наказывает находки. Естественный
вопрос: насколько сильно? На него есть измеримый ответ, и процедура его получения поучительнее
самого числа.

**Leave-one-run-out.** Коллекция TREC собрана пулингом выдач `k` систем-участниц. Берём одну
систему, **выбрасываем её вклад из пула** и пересчитываем её же метрику по обеднённой разметке.
Разница между «с её вкладом» и «без» — прямая оценка того, насколько система выигрывает
от собственного присутствия в пуле. Именно так проверяют, годится ли старая коллекция
для оценки новой системы.

**Что получается.** На классических TREC-коллекциях падение метрики у выброшенной системы
обычно единицы процентов — то есть **ранжирование систем между собой сохраняется**, даже когда
абсолютные значения смещены. Отсюда главный вывод, ради которого всё это делалось: коллекции
с пулингом надёжны для **сравнения**, но не для абсолютных утверждений. «Наша система даёт
nDCG 0,68» — утверждение про коллекцию. «Наша система обошла базовую на 0,04 на этой коллекции» —
утверждение, которое переносится.

**Где это ломается.** Ломается ровно тогда, когда новая система устроена принципиально иначе,
чем все, кто был в пуле. Плотный поиск на коллекциях, пулинг которых собран лексическими
системами, находит документы, которых никто не судил, — и получает за них ноль. Первые работы
по плотному поиску столкнулись с этим прямо, и часть их выигрыша была недооценена по построению.

**Что делать практически.** Если ты меряешь новый класс систем на старой коллекции, досуживай:
возьми топ-20 своей системы, отдай на разметку то, чего нет в qrels, и покажи метрику до
и после. Разница и есть твоя поправка на пулинг. Не показать её — значит согласиться на
неизвестное смещение не в свою пользу.
</details>

<details><summary>Сколько запросов нужно коллекции — и почему пятидесяти хватает, а пятнадцати нет</summary>

Мы всё время говорили про число запросов как про то, что определяет чувствительность. Есть
измеренный ориентир, и он старше большинства современных моделей.

**Откуда берётся 50.** Классические дорожки TREC используют 50 тем на коллекцию. Число выбрано
не из красоты: Бакли и Вурхис в начале двухтысячных мерили, при каком числе запросов
ранжирование систем становится устойчивым, — то есть при повторном отборе другого набора тем
из той же популяции порядок систем сохраняется. Пятьдесят даёт частоту ошибки в сравнении
порядка нескольких процентов при разнице в 0,05 по MAP. Меньше — растёт быстро.

**Как это связано с нашим интервалом.** Половинная ширина падает как `1/√n`. При пятнадцати
запросах и `sd = 0,068` она равна 0,037 — почти столько же, сколько сам эффект. При пятидесяти
была бы 0,019, при ста — 0,013. Отсюда практическое: **эффект 0,04 требует порядка пятидесяти
запросов**, эффект 0,01 — порядка восьмисот. Второе обычно означает, что эффект не стоит
измерения.

**Что важнее числа запросов.** Их разнообразие. Пятьдесят запросов одного типа дадут узкий
интервал и вывод, верный только для этого типа. Двадцать запросов, покрывающих навигационные,
информационные и транзакционные, дадут интервал шире, но честнее. Ошибка «набрали много
однотипных запросов» встречается чаще, чем «набрали мало».

**Проверка, которую стоит делать всегда.** Разбей свой набор запросов пополам случайно, посчитай
разницу систем отдельно на половинах. Если знак совпал — набор достаточно однороден для выводов.
Если разошёлся — у тебя не одна задача, а две, и мерить их одним средним нельзя. Это дешёвая
процедура, занимающая одну ячейку, и она снимает большинство споров о том, можно ли верить
результату.
</details>

⚠️ Ловушка A · **Категория — это не релевантность.** Подстановка «та же категория = релевантен»
даёт разметку даром и систематически завышает всё: внутри категории есть документы, никак
не отвечающие на запрос, и они считаются попаданиями. Настоящие qrels собираются людьми,
стоят дорого и всё равно неидеальны. Мы используем прокси сознательно и говорим об этом вслух —
именно это отличает упрощение от подлога.

⚠️ Ловушка A · **Непросуженное считается нерелевантным.** В нашей игрушке просужены все восемь
документов. В любой реальной коллекции судят только то, что попало в пул выдач нескольких
систем (pooling), а остальное по умолчанию нерелевантно. Отсюда: **новая система, которая
находит хорошие документы, не попавшие ни в один пул, будет наказана метрикой за находку.**
Это не гипотетика, а известное свойство TREC-коллекций.

In [ ]:
if INDEX_PATH.exists():
    IDX = json.load(open(INDEX_PATH, encoding="utf-8"))
    print(f"индекс недели 3 найден: {IDX['N']} документов, {len(IDX['df'])} терминов, "
          f"avgdl {IDX['avgdl']:.1f}")
    print("но qrels для этого корпуса у нас НЕТ -- ни одного суждения о релевантности")
    print(f"измеримо на нём сегодня: ничего из метрик ниже. "
          f"Считаем на {N_DOCS} документах лекции, где разметка есть.")
else:
    print("индекса недели 3 нет -- работаем на разметке лекции, как и планировалось")
    print(f"документов с разметкой: {N_DOCS}")
RUN["index_from_week3"] = INDEX_PATH.exists()

**Что видно.** Индекс с прошлого занятия либо есть, либо нет, и **в обоих случаях метрики
сегодня считаются не на нём**. Сравнивать надо не размеры двух корпусов, а **наличие разметки**:
у тысячи с лишним документов недели 3 нет ни одного суждения о релевантности, а у восьми
документов лекции есть. Механизм прямой и есть главный ответ на вопрос, почему на прошлом
занятии мы ни разу не сказали «лучше»: без qrels метрику посчитать не к чему, сколько бы
документов ты ни проиндексировал. Чего этот вывод НЕ показывает: что большой корпус бесполезен —
он понадобится на неделе 7, когда разметку мы к тому времени соберём. Что делать: запомнить
асимметрию. Проиндексировать миллион документов стоит часы машинного времени; просудить
пятьдесят запросов по ним стоит недели человеческого.

### Шаг 1.2 · Замер без модели: сколько стоит разметка

**Замер без модели.** Прежде чем считать метрики, посчитаем, во что обходится сама разметка
и что она физически может покрыть. Ни одна модель здесь не участвует — это свойство
процедуры, а не системы.

In [ ]:
for n_docs, n_queries, per_doc_sec in [(10_000, 50, 30), (1_000_000, 50, 30)]:
    full = n_docs * n_queries * per_doc_sec / 3600
    pooled = 100 * n_queries * per_doc_sec / 3600     # пул: топ-100 от каждой системы
    print(f"корпус {n_docs:>9,} док · {n_queries} запросов: "
          f"полная разметка {full:>9,.0f} ч · пул топ-100 {pooled:>5,.0f} ч "
          f"· покрытие {100 * n_queries / (n_docs * n_queries):.4%}")
RUN["judging_cost_note"] = "полная разметка миллионного корпуса нереализуема -- отсюда pooling"

**Что видно.** Полная разметка миллионного корпуса по пятидесяти запросам — это сотни тысяч
человеко-часов, то есть она невозможна ни за какие деньги. Сравнивать надо не два числа часов
друг с другом, а **покрытие с единицей**: пул из топ-100 трогает доли процента корпуса.
Механизм отсюда прямой — раз судить всё нельзя, судят объединение выдач нескольких систем,
и всё остальное объявляется нерелевантным по умолчанию. Чего этот расчёт НЕ показывает:
насколько велика ошибка от такого допущения; она зависит от того, насколько системы в пуле
похожи друг на друга. Что делать: при чтении любой работы с метриками на TREC-подобной
коллекции первым делом искать, из каких систем собран пул. Если твоя система непохожа
на них, твоя полнота занижена по построению.

### Шаг 1.3 · Согласие судей

Если два человека размечают одно и то же и расходятся, метрика унаследует их расхождение.
Считаем каппу Коэна на синтетике — без обучения, без модели, чистая арифметика.

In [ ]:
def cohen_kappa(a, b):
    n = len(a)
    po = sum(x == y for x, y in zip(a, b)) / n
    pa1, pb1 = sum(a) / n, sum(b) / n
    pe = pa1 * pb1 + (1 - pa1) * (1 - pb1)
    return (po - pe) / (1 - pe), po, pe

rng = np.random.default_rng(SEED)
truth = rng.integers(0, 2, 200)
for noise in (0.0, 0.1, 0.25):
    j1 = [t if rng.random() > noise else 1 - t for t in truth]
    j2 = [t if rng.random() > noise else 1 - t for t in truth]
    k, po, pe = cohen_kappa(j1, j2)
    print(f"шум судьи {noise:>4.0%}: согласие {po:.2f} · случайное {pe:.2f} · каппа {k:.2f}")

**Что видно.** Сравнивать надо не «согласие» само по себе, а **согласие с колонкой случайного**:
при бинарной разметке два судьи, бросающие монетку, совпадут примерно в половине случаев,
и голое «согласие 0,5» означает полное отсутствие сигнала. Каппа вычитает эту базу, поэтому
она и есть правильное число. Ожидаемая картина по механизму: с ростом шума согласие падает
медленно, а каппа — быстро, потому что случайная база остаётся на месте. Чего этот замер
НЕ показывает: систематического расхождения — если оба судьи одинаково неправы, каппа будет
высокой, а разметка плохой. Что делать: требовать каппу, а не процент согласия, и помнить,
что каппа около 0,6 в TREC-разметке считается нормой, то есть **пятая часть суждений спорна
даже у профессионалов**.

<details><summary>Какая именно каппа — и почему для градуированной шкалы обычная не годится</summary>

«Каппа» — не одна метрика, а семейство, и выбор внутри него меняет число заметно.

**Каппа Коэна** считается для **двух** судей и номинальной шкалы. Именно её мы посчитали.
Она вычитает согласие, ожидаемое при независимом угадывании с наблюдаемыми частотами меток.

**Каппа Фляйсса** обобщает на произвольное число судей, причём судьи не обязаны быть одними
и теми же для всех объектов, — что как раз соответствует реальной разметке, где пул асессоров
велик, а каждый документ смотрят двое-трое.

**Взвешенная каппа** — то, что нужно для шкалы 0/1/3 из части 3. Обычная каппа считает
расхождение «0 против 3» и «1 против 3» одинаково плохими, хотя первое — грубая ошибка,
а второе — вопрос вкуса. Взвешенная вводит матрицу штрафов; при квадратичных весах она
совпадает с внутриклассовой корреляцией и ведёт себя разумно.

**Известная неприятность: парадокс каппы.** При сильном перекосе классов каппа может быть
низкой при очень высоком согласии. Если релевантны 2 % документов и оба судьи согласны
в 97 % случаев, каппа окажется около 0,23 — формально «слабое согласие», хотя судьи совпали
в 97 % случаев: случайное согласие при таком перекосе само по себе почти 96 %. В поиске перекос именно такой, и потому каппу на qrels надо читать
осторожно, а лучше рядом с ней показывать долю согласия отдельно по релевантным.

**Практический ориентир.** В TREC согласие асессоров исторически даёт каппу порядка 0,5–0,6.
Это значит, что около пятой части суждений спорна у профессионалов. Любая разница между
системами меньше этой неопределённости — не разница, а вопрос о том, чей асессор размечал.
</details>

⚠️ Ловушка C · **Высокая каппа не означает правильной разметки.** Она означает воспроизводимость
суждений, а не их истинность. Два судьи с одинаковым неверным пониманием запроса дадут каппу
0,9 и разметку, по которой любая система будет оптимизироваться не туда.

---

## Часть 2 · Метрики по позициям — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 2.1 | Чему равна метрика у случайного порядка? | Монте-Карло: наше нулевое число |
| 2.2 | Recall@k и Precision@k | реализуем и сверяем с `data/l4-metrics.json` |
| 2.3 | Чем MRR отличается от MAP? | считаем на двух запросах, где они расходятся |

### Шаг 2.1 · Число, с которым сравнивается всё остальное

nDCG равен 0,68 — это хорошо или плохо? Вопрос бессмысленный, пока нет базы. Простейшая база
для ранжирования — **случайный порядок**: он не смотрит в документы вообще.

In [ ]:
def dcg(rels):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))

def ndcg(rels):
    ideal = dcg(sorted(rels, reverse=True))
    return dcg(rels) / ideal if ideal else 0.0

pool = list(RELS)
samples = []
for _ in range(N_TRIALS):
    random.shuffle(pool)
    samples.append(ndcg(pool))

BASE = statistics.mean(samples)
BASE_SD = statistics.pstdev(samples)
print(f"nDCG случайного порядка: среднее {BASE:.4f} · разброс {BASE_SD:.4f} "
      f"· 5-й и 95-й перцентили {np.percentile(samples, 5):.3f}--{np.percentile(samples, 95):.3f}")
print(f"nDCG нашего BM25-порядка (из лекции): {M['ndcg']:.4f}")
RUN["base_random_ndcg"], RUN["ndcg_bm25"] = BASE, M["ndcg"]

# Насколько логарифмический дисконт падает между СОСЕДНИМИ позициями. Проза ниже
# сравнивает первую пару с девятой; числа считаем, а не заявляем.
_disc = lambda i: 1 / math.log2(i + 1)
RUN["discount_drop_pct"] = {f"{i}->{i+1}": round((1 - _disc(i + 1) / _disc(i)) * 100, 1)
                            for i in (1, 9)}
print("падение дисконта: " + " · ".join(f"{k} = {v} %" for k, v in RUN["discount_drop_pct"].items()))

# Парадокс каппы: при сильном перекосе классов случайное согласие само по себе велико,
# и высокая доля совпадений даёт низкую каппу. Считаем на примере из разбора.
# Границы бутстрэп-интервала, о которых говорит разбор («взять 2,5-й и 97,5-й
# перцентили») — объявляем их числами, а не только словами.
RUN["ci_percentiles"] = [2.5, 97.5]
# Множественные сравнения: при уровне 0,05 на КАЖДУЮ проверку вероятность хотя бы одного
# ложного «значимо» растёт как 1 − (1−α)^k. Разбор ниже приводит k = 10 и k = 20 —
# считаем оба, чтобы число в тексте держалось на расчёте, а не на памяти.
RUN["alpha"] = 0.05
RUN["family_error_pct"] = {str(k): round((1 - (1 - RUN["alpha"]) ** k) * 100, 1)
                           for k in (10, 20)}
print("вероятность хотя бы одного ложного «значимо»: " + " · ".join(
    f"{k} конфигураций → {v} %" for k, v in RUN["family_error_pct"].items()))
_rel_share, _observed = 0.02, 0.97
_pe = _rel_share ** 2 + (1 - _rel_share) ** 2
RUN["kappa_paradox"] = {"rel_share_pct": _rel_share * 100, "observed_agree_pct": _observed * 100,
                        "chance_agree_pct": round(_pe * 100, 1),
                        "kappa": round((_observed - _pe) / (1 - _pe), 3)}
print(f"парадокс каппы: судьи согласны в {_observed:.0%}, но случайное согласие уже "
      f"{_pe:.1%} → каппа {RUN['kappa_paradox']['kappa']}")

**Что видно.** Главное — не величина базы, а её положение относительно BM25: случайный порядок
даёт в среднем **больше**, чем наш BM25. Сравнивать надо именно так, и вывод неприятный:
на этой коллекции лексический ранкер проигрывает перемешиванию. Механизм не мистический
и стоит того, чтобы его назвать: релевантность у нас выведена из категории 20NG, а BM25
ранжирует по совпадению слов, и на первое место он поставил документ про хоккей, где слово
`team` встречается часто. Прокси-разметка и ранкер расходятся, и расходятся сильно.
Второе: при половине релевантных из восьми любой порядок собирает много — дисконт мягкий,
идеальный порядок недалеко. Чего этот замер НЕ показывает: что BM25 плох. Он показывает, что
**наша разметка не измеряет то, что оптимизирует BM25**. Что делать: запомни `BASE` и не цитируй
ни одного nDCG без него. И держи в голове, что метрика ниже случайной — это чаще всего
диагноз разметке, а не системе.

<details><summary>Откуда взялся логарифмический дисконт — и три альтернативы, которые лучше моделируют человека</summary>

Дисконт `1/log2(rank+1)` в DCG — соглашение, а не измеренная физика. Джарвелин и Кекяляйнен
в 2002 году выбрали его за два свойства: он монотонно убывает и убывает **медленно**, так что
позиции за пределами первой пятёрки всё-таки вносят вклад. Никакой модели поведения
пользователя за ним нет.

**RBP (rank-biased precision).** Модель явная: пользователь смотрит документ и с вероятностью
`p` переходит к следующему. Вес позиции равен `p^(rank-1)`, метрика равна `(1-p)·Σ rel·p^(rank-1)`.
Параметр `p` — это «терпеливость», и его можно **оценить из логов**, а не назначить. Заодно
RBP корректно ведёт себя при неполной разметке: непросуженные документы дают интервал
неопределённости, а не ноль.

**ERR (expected reciprocal rank).** Каскадная модель: пользователь идёт вниз, пока не находит
устраивающий документ, и вероятность остановки растёт с градацией релевантности. Метрика равна
ожидаемому обратному рангу остановки. Она сильнее наказывает мусор наверху, чем nDCG, и лучше
коррелирует с кликами.

**Геометрический просмотр из наших данных.** В `data/l4-online.json` лежит модель
`P(exam|rank) = γ^(rank-1)` со своим γ. Сравни её с логарифмическим дисконтом: на десятой
позиции геометрическая даёт 0,57, логарифмическая — 0,29. То есть nDCG считает десятую позицию
вдвое менее важной, чем измеренная вероятность просмотра. Дисконт nDCG **пессимистичнее**
реального поведения.

**Что из этого следует.** Не «nDCG плохая». Следует, что при выборе между двумя системами,
различающимися на дальних позициях, nDCG занизит разницу относительно того, что заметит
пользователь. Если твой продукт показывает десять результатов и пользователь реально их
листает, ERR или RBP с оценённым из логов `p` ближе к делу.
</details>

⚠️ Ловушка B · **nDCG на маленькой коллекции завышен и почти нечувствителен.** Чем меньше
документов и чем больше доля релевантных, тем ближе любой порядок к идеальному. Именно поэтому
метрики считают на коллекциях в тысячи документов и на десятках запросов, а не на восьми.
Наши сегодняшние числа воспроизводят **явление**, а не величину.

In [ ]:
plt.figure(figsize=(9, 3))
plt.hist(samples, bins=40, color="#3B6FD4", alpha=.85)
plt.axvline(BASE, color="#111", ls="--", label=f"случайный порядок {BASE:.3f}")
plt.axvline(M["ndcg"], color="#B4521F", lw=2, label=f"BM25 {M['ndcg']:.3f}")
plt.axvline(1.0, color="#2E7D52", lw=2, label="идеальный порядок 1.0")
plt.xlabel("nDCG"); plt.ylabel("прогонов"); plt.legend()
plt.title("Наше число на фоне случайного: сравнивать надо с распределением, а не с нулём")
plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо положение оранжевой линии **относительно массы гистограммы**,
а не относительно левого края графика. BM25 сидит в левой части распределения случайных
порядков — то есть заметная доля случайных перестановок оказывается лучше него. Ожидаемая
картина именно такая по механизму из предыдущего разбора: разметка и ранкер меряют разное.
Чего гистограмма НЕ показывает: поведения на большой коллекции — там распределение случайного
nDCG сожмётся к маленькому значению, и любая осмысленная система уйдёт далеко вправо. Что
делать: всякий раз, когда видишь одиночное число метрики, мысленно дорисовывай под ним это
распределение. Без него число — украшение, а иногда и опровержение того, что им доказывают.

### Шаг 2.2 · Recall@k и Precision@k

Две метрики, которые чаще всего путают, и разница между ними — это разница между «не потеряли»
и «не насыпали мусора».

In [ ]:
def recall_at_k(rels, k, total_rel):
    return sum(rels[:k]) / total_rel if total_rel else 0.0

def precision_at_k(rels, k):
    return sum(rels[:k]) / k if k else 0.0

bad = []
for k in M["ks"]:
    r, p = recall_at_k(RELS, k, N_REL), precision_at_k(RELS, k)
    if abs(r - M["recallAtK"][str(k)]) > 1e-4 or abs(p - M["precisionAtK"][str(k)]) > 1e-4:
        bad.append((k, r, p))
assert not bad, f"расхождение с лекцией: {bad}"

print(f"{'k':>3} {'Recall@k':>10} {'Precision@k':>12}")
for k in M["ks"]:
    print(f"{k:>3} {recall_at_k(RELS, k, N_REL):>10.4f} {precision_at_k(RELS, k):>12.4f}")
print(f"сверка с data/l4-metrics.json: {2 * len(M['ks'])} значений совпали")

**Что видно.** Сравнивать надо не два столбца друг с другом, а **их поведение с ростом k**:
Recall монотонно растёт до единицы, Precision — падает. Это не свойство нашей системы, а
арифметика: знаменатель Recall фиксирован (число релевантных), знаменатель Precision растёт
вместе с k. Механизм означает, что «улучшить Recall@k, увеличив k» можно всегда и это ничего
не стоит. Чего эта таблица НЕ показывает: что происходит между значениями k — мы посчитали
четыре точки, а кривая между ними может вести себя как угодно. Что делать: никогда не называть
Recall без k и без Precision рядом. «Recall 100 %» на выдаче из всего корпуса — это правда
и бессмыслица одновременно.

⚠️ Ловушка B · **Высокий Recall@100 и три ссылки на экране.** Метрика отвечает на вопрос
«попало ли нужное в кандидаты», и для первой ступени каскада это правильный вопрос. Для
продукта, где пользователь видит три результата, правильный вопрос — Precision@3, и он может
быть ужасен при великолепном Recall@100. Обе метрики верны; неверно называть одну из них
«качеством поиска».

### Шаг 2.3 · MRR и MAP на двух запросах

На одном запросе MRR и RR совпадают, MAP и AP тоже — и различие невидимо. Берём два.

In [ ]:
MQ = json.load(open(f"{DATA_DIR}/l4-multiquery.json", encoding="utf-8"))

def rr(rels):
    for i, r in enumerate(rels, 1):
        if r:
            return 1 / i
    return 0.0

def ap(rels):
    hits, s = 0, 0.0
    for i, r in enumerate(rels, 1):
        if r:
            hits += 1
            s += hits / i
    return s / hits if hits else 0.0

qs = [MQ["q1"]["rels"], MQ["q2"]["rels"]]
assert abs(rr(qs[0]) - MQ["q1"]["rr"]) < 1e-4 and abs(ap(qs[0]) - MQ["q1"]["ap"]) < 1e-4
assert abs(rr(qs[1]) - MQ["q2"]["rr"]) < 1e-4 and abs(ap(qs[1]) - MQ["q2"]["ap"]) < 1e-4

mrr = statistics.mean(rr(q) for q in qs)
mapv = statistics.mean(ap(q) for q in qs)
print(f"{'запрос':>8} {'релевантность':>26} {'RR':>7} {'AP':>7}")
for i, q in enumerate(qs, 1):
    print(f"{'Q' + str(i):>8} {str(q):>26} {rr(q):>7.4f} {ap(q):>7.4f}")
print(f"{'MRR':>8} {'':>26} {mrr:>7.4f}")
print(f"{'MAP':>8} {'':>26} {'':>7} {mapv:>7.4f}")
assert abs(mrr - MQ["mrr"]) < 1e-4 and abs(mapv - MQ["map"]) < 1e-4, "MRR/MAP разошлись с лекцией"
RUN["mrr"], RUN["map"] = mrr, mapv

**Что видно.** У Q2 первый документ релевантен, поэтому RR равен единице — максимум, независимо
от того, что дальше. У Q1 первый релевантный на втором месте, RR равен половине. Сравнивать надо
не RR с AP по величине, а **их чувствительность**: RR смотрит ровно на одну позицию и слеп
ко всему остальному, AP учитывает каждое попадание. Механизм виден в формулах: у RR в сумме
одно слагаемое, у AP — по слагаемому на каждый релевантный документ. Чего эти числа НЕ
показывают: какая метрика «правильная» — это зависит от продукта. Что делать: выбирать MRR,
когда пользователю нужен один ответ (навигационный запрос, поиск по документации), и MAP,
когда ему нужен полный набор (обзор литературы, подбор товаров).

⚠️ Ловушка B · **MRR не замечает ничего после первого попадания.** Система, которая ставит
один идеальный документ первым, а дальше девять мусорных, и система с десятью идеальными
документами получат **одинаковый** MRR. Если это не то, что ты имел в виду, MRR — не твоя метрика.

⚠️ Ловушка D · **Усреднять надо по запросам, а не по документам.** Если сложить все попадания
всех запросов и поделить на общее число позиций, запрос с длинной выдачей перевесит запрос
с короткой, и метрика станет средневзвешенной по объёму, а не по пользователям. Мы усредняем
через `statistics.mean` по списку запросов — это макро-усреднение, и именно оно принято.

---

## Часть 3 · Градуированная релевантность и nDCG — 25 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 3.1 | Зачем градации, если есть бинарная метка? | считаем DCG на шкале 0/1/3 |
| 3.2 | Линейный выигрыш или экспоненциальный? | считаем оба и сравниваем разницу с разбросом |
| 3.3 | Что именно делает дисконт? | смотрим на вклад каждой позиции |

Бинарная метка отвечает «да/нет». Реальность отвечает «отлично / сойдёт / мимо», и разница
между «отлично» и «сойдёт» на первой позиции стоит дороже, чем на десятой.

In [ ]:
G = json.load(open(f"{DATA_DIR}/l4-graded.json", encoding="utf-8"))
grades = G["gainsInRankOrder"]
ideal = G["idealGains"]

def dcg_gain(gains):
    return sum(g / math.log2(i + 2) for i, g in enumerate(gains))

def ndcg_gain(gains, ideal_gains):
    idcg = dcg_gain(ideal_gains)
    return dcg_gain(gains) / idcg if idcg else 0.0

lin = ndcg_gain(grades, ideal)
exp_gains = [2 ** g - 1 for g in grades]
exp_ideal = [2 ** g - 1 for g in ideal]
exp = ndcg_gain(exp_gains, exp_ideal)

assert abs(dcg_gain(grades) - G["linear"]["dcg"]) < 1e-3, "линейный DCG разошёлся с лекцией"
assert abs(lin - G["linear"]["ndcg"]) < 1e-3, "линейный nDCG разошёлся с лекцией"
assert abs(exp - G["exponential"]["ndcg"]) < 1e-3, "экспоненциальный nDCG разошёлся с лекцией"

print("шкала оценок:", G["gradeScale"])
print("оценки в порядке выдачи:", grades, "· идеальный порядок:", ideal)
print(f"nDCG линейный        {lin:.4f}   (DCG {dcg_gain(grades):.4f} / IDCG {dcg_gain(ideal):.4f})")
print(f"nDCG экспоненциальный {exp:.4f}   (2^rel - 1)")
print(f"разница: {abs(lin - exp):.4f}")
RUN["ndcg_linear"], RUN["ndcg_exp"] = lin, exp

**Что видно.** Две формулы выигрыша дали nDCG, отличающиеся в третьем знаке. Сравнивать надо
не значения друг с другом, а **их разницу с разбросом случайного порядка**, который мы померяли
в части 2: разброс там был в сотых, разница здесь — в тысячных. То есть на этой коллекции выбор
между линейным и экспоненциальным выигрышем **не важен**. Ожидаемая картина по механизму:
экспонента раздувает и числитель, и знаменатель, и при трёх градациях эффект почти сокращается.
Чего этот замер НЕ показывает: поведения при большем числе градаций и сильном перекосе — на
шкале 0–4 с редкими четвёрками экспонента расходится с линейной заметно. Что делать: не тратить
время на выбор формулы выигрыша, пока не показал, что он больше твоего разброса. Это тоже
результат: он экономит день.

<details><summary>Как обрезают nDCG@k разные библиотеки — и почему числа не сходятся</summary>

Обрезка nDCG по позиции `k` кажется тривиальной. На ней расходятся практически все реализации,
и разница доходит до сотых.

**Развилка первая: чем обрезать IDCG.** Правильно — идеальным порядком, **обрезанным по тому
же `k`**. Встречается и вариант, где IDCG считается по всей выдаче: тогда nDCG@k занижен
и не достигает единицы даже у идеального ранжирования. В задании 1 первый `assert` ловит
ровно эту ошибку.

**Развилка вторая: что делать, если релевантных меньше `k`.** Идеальный порядок обрезается
по `k`, но релевантных всего три при `k = 10` — тогда IDCG@10 = DCG идеального из трёх. Это
корректно. Ошибка возникает, когда реализация дополняет идеальный порядок нулями иначе, чем
фактический.

**Развилка третья: `sklearn.metrics.ndcg_score`.** Он принимает **матрицу оценок**, а не список
релевантностей в порядке выдачи, и сам сортирует по переданным скорам. Если передать
релевантности как скоры, получишь nDCG идеального порядка, то есть единицу, и не заметишь.
Кроме того, `ndcg_score` по умолчанию использует линейный выигрыш и требует неотрицательных
оценок; экспоненциальный надо возводить руками до вызова.

**Развилка четвёртая: пустые и вырожденные случаи.** При нулевом IDCG (ни одного релевантного)
одни реализации возвращают ноль, другие — единицу, третьи — NaN. Усреднение по запросам после
этого даёт три разных числа на одних данных.

**Что делать.** Ровно то, что мы сделали: своя реализация в двадцать строк, сверенная
с зафиксированными числами. Она заведомо совпадает сама с собой на всём курсе, и это важнее,
чем совпадать с чужой библиотекой. Единственное число, которое можно сравнивать с чужим, —
то, где обе стороны опубликовали формулу.
</details>

<details><summary>Что теряется при нормировке nDCG — и когда его нельзя усреднять по запросам</summary>

Буква N в nDCG означает деление на IDCG — на DCG идеального порядка. Это делает метрику
сравнимой между запросами, и это же прячет важное.

**Что нормировка даёт.** Запрос с двадцатью релевантными документами и запрос с одним
несравнимы по сырому DCG: у первого потолок выше просто потому, что складывать есть что.
После деления на собственный потолок оба живут в `[0,1]`, и их можно усреднять.

**Что она забирает.** Информацию о сложности запроса. nDCG = 0,9 на запросе с одним релевантным
документом означает «нашли его на первом-втором месте». nDCG = 0,9 на запросе с двадцатью
означает «собрали почти идеальный порядок из двадцати». Это очень разные достижения, и после
усреднения они весят одинаково.

**Когда усреднять нельзя.** Если в наборе есть запросы **без единого релевантного документа**,
IDCG равен нулю и nDCG не определён. Обычная практика — выбросить такие запросы, и это молча
меняет набор: выброшенные обычно самые трудные. Второй случай — сильно разный размер выдачи
у разных запросов при обрезке по `k`: nDCG@10 у запроса, где всего три документа, считается
по другому потолку, и сравнение с запросом на тысячу кандидатов условно.

**Практический приём.** Показывать рядом с усреднённым nDCG **распределение по запросам**,
а не только среднее, — гистограмму или хотя бы квартили. Именно так в части 4 мы увидели,
что на четырёх запросах из пятнадцати знак разницы меняется. Среднее это скрывало.

**Про наш случай.** У нас один запрос и восемь документов, из которых четыре релевантны.
Это худший возможный режим для nDCG: потолок близко, любой порядок неплох. Все выводы
сегодняшнего занятия о **процедуре** верны, все выводы о **величинах** — нет.
</details>

⚠️ Ловушка C · **«Разница в третьем знаке» — не «формулы эквивалентны».** Мы показали это
на одном запросе и восьми документах. Правильный вывод — «на нашей коллекции разница внутри
шума», а не «экспонента бесполезна». Второе утверждение потребовало бы замера на многих
коллекциях, которого мы не делали.

In [ ]:
disc = [1 / math.log2(i + 2) for i in range(10)]
plt.figure(figsize=(9, 3))
plt.bar(range(1, 11), disc, color="#3B6FD4")
for i, d in enumerate(disc[:5], 1):
    plt.text(i, d + .02, f"{d:.2f}", ha="center", fontsize=9)
plt.xlabel("позиция в выдаче"); plt.ylabel("вес позиции 1/log2(rank+1)")
plt.title("Дисконт: вторая позиция стоит 0.63 первой, десятая -- 0.29")
plt.xticks(range(1, 11)); plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо не столбики попарно, а **первый столбик со всеми остальными**:
падение с первой позиции на вторую — это минус 37 %, а с девятой на десятую — минус 4 %.
Логарифмический дисконт устроен так по механизму: он моделирует убывающее внимание
пользователя, и модель эта грубая, но лучше равномерной. Чего этот график НЕ показывает:
реального внимания реальных пользователей — оно измеряется отдельно, и в `data/l4-online.json`
лежит геометрическая модель со своим γ, которая падает совсем иначе. Что делать: помнить,
что дисконт nDCG — соглашение, а не физика. Две системы, которые расходятся только на восьмой
позиции, метрика почти не различит, а пользователь — тем более.

<details><summary>Офлайн против онлайна: что меряют A/B и чередование, и почему они расходятся с nDCG</summary>

Всё сегодняшнее занятие — офлайн-оценка: есть разметка, есть выдачи, метрика считается без
пользователей. У неё есть предел, и в `data/l4-online.json` лежат числа второго подхода.

**A/B-тест.** Пользователи делятся случайно, половина видит контроль, половина — новую систему.
В данных: 10 000 на группу, CTR 12,0 % против 13,2 %, `z = 2,56`, `p = 0,011`, относительный
прирост 10 %. Это прямое измерение поведения, и оно перекрывает любые офлайн-метрики
по убедительности. Цена: недели ожидания, риск показать плохую выдачу реальным людям
и требование трафика, которого у большинства систем нет.

**Чередование (interleaving).** Хитрее и в разы чувствительнее: одному пользователю показывают
**смешанную** выдачу из обеих систем и смотрят, чьи документы кликают. Каждый пользователь
становится своим собственным контролем, межпользовательский разброс уходит, и для того же
вывода нужно на порядок меньше трафика. Именно так сравнивают ранжирующие модели в поиске
на практике.

**Смещение позиции.** Главная ловушка кликов: верхние результаты кликают чаще просто потому,
что они верхние. Геометрическая модель из данных даёт вероятность просмотра `γ^(rank-1)`.
Без поправки на это любая метрика по кликам будет измерять позицию, а не релевантность,
и система, поставившая случайный документ первым, покажет отличный CTR.

**Почему офлайн и онлайн расходятся.** Офлайн меряет соответствие разметке, онлайн — поведение.
Расходятся они систематически: разметчик оценивает документ «по существу», а пользователь
кликает на понятный заголовок. Улучшение nDCG на 0,04 может не дать никакого прироста CTR,
и это не ошибка ни одного из измерений. Правильный порядок — офлайн отсеивает заведомо плохое
дёшево, онлайн решает окончательно и дорого.
</details>

---

## Часть 4 · Система A лучше B, или просто повезло? — 30 мин

| шаг | вопрос | чем отвечаем |
|---|---|---|
| 4.1 | На сколько B выигрывает в среднем? | считаем среднее по 15 запросам |
| 4.2 | Насколько устойчив этот выигрыш? | смотрим на знак разницы по каждому запросу |
| 4.3 | Что говорит статистика? | парный t-тест, доверительный интервал, перестановочный тест |

Это самая важная часть занятия. Всё остальное — арифметика; здесь принимается решение.

In [ ]:
S = json.load(open(f"{DATA_DIR}/l4-systems.json", encoding="utf-8"))
A, B = S["systemA"], S["systemB"]
diffs = [round(b - a, 4) for a, b in zip(A, B)]

assert diffs == S["perQueryDiff"], "поразрядные разницы разошлись с лекцией"
print(f"запросов: {len(A)}")
print(f"среднее A: {statistics.mean(A):.4f} · среднее B: {statistics.mean(B):.4f} "
      f"· разница {statistics.mean(diffs):+.4f}")
print(f"B выигрывает на {sum(d > 0 for d in diffs)} запросах, проигрывает на "
      f"{sum(d < 0 for d in diffs)}")
RUN["mean_diff"] = statistics.mean(diffs)

**Что видно.** B в среднем лучше на четыре сотых nDCG, и это выглядит как победа. Сравнивать
надо не средние друг с другом, а **среднее с числом проигрышей**: B проигрывает на четырёх
запросах из пятнадцати, то есть больше чем в четверти случаев. Механизм важен: среднее по
запросам скрывает, что эффект неоднороден — где-то B выигрывает сильно, где-то устойчиво
проигрывает. Чего эти два числа НЕ показывают: случаен ли перевес; для этого нужен разброс,
и он в следующей ячейке. Что делать: никогда не останавливаться на среднем. «B лучше на 4 %» —
это ровно та формулировка, за которой прячется всё интересное.

<details><summary>Разброс между запросами против разброса внутри запроса — почему парный дизайн решает всё</summary>

Посмотри на числа: средние систем 0,605 и 0,644, а отдельные значения гуляют от 0,49 до 0,86.
Разброс **между запросами** в разы больше разницы **между системами**. Это не особенность
наших данных, а универсальное свойство поиска.

**Что было бы при непарном сравнении.** Если бы мы мерили A на одних пятнадцати запросах,
а B — на других пятнадцати, стандартная ошибка разницы считалась бы из разброса самих значений:
`sd(A)` около 0,09, `sd(B)` около 0,09, и SE разницы вышла бы около 0,033. Наблюдаемая разница
0,0397 дала бы `t ≈ 1,2` и `p ≈ 0,25` — «ничего не обнаружено».

**Что даёт парный дизайн.** Мы считаем разницу **на каждом запросе** и работаем с ней. Разброс
разниц `sd = 0,068`, SE = 0,0175 — вдвое меньше. Тот же эффект стал различим. Механизм: трудный
запрос труден для обеих систем, и вычитание убирает эту общую компоненту целиком.

**Цена.** Парный дизайн требует, чтобы обе системы прогонялись на **одном и том же** наборе
запросов и с одной и той же разметкой. Звучит очевидно, нарушается постоянно: кто-то досудил
пул под новую систему, кто-то выкинул запросы без релевантных, кто-то поменял версию коллекции.
После этого пары перестают быть парами, а формула остаётся прежней и выдаёт число.

**Как проверить, что дизайн действительно парный.** Первый `assert` в ячейке части 4 сравнивает
поразрядные разницы с числами лекции — это заодно проверка, что порядок запросов в обоих списках
один и тот же. Перепутанный порядок дал бы правдоподобные, но бессмысленные разницы, не выбросив
никакой ошибки. Это ловушка типа D в чистом виде: код работает, результат тихо неверен.

**Следствие для планирования.** Если тебе нужно сравнить пять конфигураций, прогоняй все пять
на одном наборе запросов сразу, а не по мере готовности. Иначе ты потеряешь парность и вместе
с ней втрое больше чувствительности, чем сможешь вернуть, добавляя запросы.
</details>

<details><summary>Предпосылки парного t-теста, что делать при их нарушении и почему перестановочный честнее</summary>

Мы посчитали три теста и получили похожие `p`. Это не случайность и не подтверждение —
это следствие того, что данные ведут себя прилично.

**Что требует парный t-тест.** Не нормальности самих метрик, а приближённой нормальности
**распределения средней разницы**. При пятнадцати наблюдениях центральная предельная теорема
работает лишь отчасти, и тест чувствителен к выбросам: один запрос с огромной разницей
раздувает и среднее, и `sd`.

**Что делать, если предпосылки под вопросом.** Ранговый тест Уилкоксона не требует нормальности,
но проверяет другую гипотезу — про симметрию распределения разниц относительно нуля, а не про
среднее. Его `p` не взаимозаменяем с `p` t-теста, хотя обычно близок.

**Почему перестановочный тест честнее всех.** Он не предполагает ничего о форме распределения.
Логика прямая: если системы неразличимы, то знак разницы на каждом запросе — результат
подбрасывания монетки. Перебираем **все** 2^15 = 32768 расстановок знаков, для каждой считаем
среднее, и смотрим, какая доля дала среднее не меньше наблюдённого. Это и есть `p`, и он
точный, а не приближённый. При пятнадцати запросах полный перебор занимает миллисекунды —
приближения не нужны вовсе.

**Что из этого следует для отчёта.** Приводить надо перестановочный `p` и доверительный
интервал. `p` отвечает «мог ли такой перевес возникнуть от перемешивания», интервал отвечает
«насколько велик эффект». Второе почти всегда важнее, и почти всегда именно его не приводят.

<summary>Как сделать правильно, если есть бюджет</summary>
Бутстрап по запросам: пересэмплировать пятнадцать запросов с возвращением десять тысяч раз,
каждый раз считать среднюю разницу, взять 2,5-й и 97,5-й перцентили. Получится интервал,
не опирающийся ни на нормальность, ни на t-таблицу, и он честно покажет, насколько узкая
у тебя выборка запросов.
</details>

In [ ]:
plt.figure(figsize=(9, 3.2))
colors = ["#2E7D52" if d > 0 else "#B4521F" for d in diffs]
plt.bar(range(1, 16), diffs, color=colors)
plt.axhline(0, color="#111", lw=1)
plt.axhline(statistics.mean(diffs), color="#3B6FD4", ls="--",
            label=f"среднее {statistics.mean(diffs):+.4f}")
plt.xlabel("запрос"); plt.ylabel("nDCG@10 B минус A"); plt.legend()
plt.title("Знак разницы меняется: на 4 запросах из 15 выигрывает A")
plt.xticks(range(1, 16)); plt.tight_layout(); plt.show()

**Что видно.** Сравнивать надо не столбики между собой, а **их знаки с пунктиром среднего**:
четыре столбика уходят вниз, и один из них по модулю сопоставим с самыми высокими зелёными.
То есть на конкретном запросе исход может быть любым, и «B лучше» — утверждение про среднее
по популяции запросов, а не про твой следующий запрос. Ожидаемая картина такая по механизму:
эффект мал относительно разброса между запросами, а запросы различаются между собой сильнее,
чем системы. Чего график НЕ показывает: почему именно на этих четырёх запросах A выигрывает —
и это самый ценный вопрос, который тут можно задать. Что делать: прежде чем катить B в прод,
посмотреть глазами на четыре проигранных запроса. Часто там обнаруживается класс запросов,
который новая система ломает системно, и среднее его прячет.

⚠️ Ловушка C · **Среднее по запросам — не «система лучше».** Это оценка среднего эффекта
на распределении запросов, из которого выбраны эти пятнадцать. Если твои реальные запросы
распределены иначе (а они распределены иначе — у них длинный хвост), число не переносится.

⚠️ Ловушка E · **Сравнивать можно только на одном наборе.** A и B здесь померяны на одних
и тех же пятнадцати запросах, и потому разницу можно считать поразрядно. Если бы наборы
отличались, единственным честным сравнением было бы сравнение средних с их доверительными
интервалами, и мощность упала бы в разы.

In [ ]:
n = len(diffs)
mean_d = statistics.mean(diffs)
sd = statistics.stdev(diffs)
se = sd / math.sqrt(n)
t = mean_d / se
t_crit = S["ciHalfWidth"]["tCrit"]
ci = (mean_d - t_crit * se, mean_d + t_crit * se)

assert abs(t - S["pairedTTest"]["t"]) < 1e-3, "t-статистика разошлась с лекцией"
assert abs(ci[0] - S["ci95"][0]) < 1e-3 and abs(ci[1] - S["ci95"][1]) < 1e-3, "ДИ разошёлся"

print(f"среднее различие {mean_d:+.4f} · sd {sd:.4f} · se {se:.4f}")
print(f"парный t = {t:.4f} при df = {n - 1}, p = {S['pairedTTest']['p']:.5f}")
print(f"95% доверительный интервал: [{ci[0]:+.4f}; {ci[1]:+.4f}]")
print(f"Уилкоксон p = {S['wilcoxon']['p']:.5f} · перестановочный p = {S['permutation']['p']:.5f}")
RUN["ci95"], RUN["p_paired"] = list(ci), S["pairedTTest"]["p"]

# Половинная ширина падает как 1/√n — проза ниже сравнивает нынешние пятнадцать запросов
# с полусотней и сотней. Считаем это здесь, а не на словах: число, на котором стоит вывод
# «эффект 0,04 требует порядка пятидесяти запросов», обязано быть в дампе.
# Для НАШИХ пятнадцати запросов множитель — t по таблице (df=14). Для гипотетических
# пятидесяти и ста берём нормальное приближение 1,96: при таком df разница между t и z
# уже в третьем знаке, а тащить сюда таблицу t ради этого незачем.
RUN["half_width"] = {str(n): round(t_crit * sd / math.sqrt(n), 4),
                     "50": round(1.96 * sd / math.sqrt(50), 4),
                     "100": round(1.96 * sd / math.sqrt(100), 4)}
print("половинная ширина: " + " · ".join(
    f"n={k} → {v}" for k, v in RUN["half_width"].items()))

**Что видно.** Все три теста дают p около 0,04 — формально «значимо». Сравнивать надо не p
с порогом 0,05, а **нижнюю границу доверительного интервала с нулём**: она равна +0,0023,
то есть данные совместимы с выигрышем в две тысячных nDCG, а это ноль в любом практическом
смысле. Механизм: при пятнадцати запросах интервал широк, и «значимо» здесь означает лишь
«знак скорее положительный», а не «эффект такой, как среднее». Чего эти числа НЕ показывают:
практической важности — статистическая значимость и полезность это разные вещи, и первая
не влечёт вторую. Что делать: сообщать интервал, а не p. Фраза «B лучше на 0,04, 95 % ДИ
[0,002; 0,077]» честна; фраза «B значимо лучше (p<0,05)» вводит в заблуждение, не будучи ложью.

<details><summary>Множественные сравнения: почему p = 0,04 при десяти конфигурациях не значит ничего</summary>

Самая дорогая ошибка в оценке поиска не в формуле метрики, а в том, сколько раз ты посмотрел.

**Механизм.** При истинном отсутствии эффекта `p < 0,05` возникает в одном случае из двадцати
по определению. Проверив десять конфигураций, ты получаешь вероятность хотя бы одного ложного
«значимо» около `1 - 0,95^10 ≈ 40 %`. Проверив двадцать — больше половины. Ты обязательно
найдёшь победителя, даже если побеждать нечему.

**Где это происходит незаметно.** Не в явном переборе сетки. Это происходит, когда ты
посмотрел на результат, чуть поменял токенизацию, посмотрел снова, потом добавил стоп-слова,
потом поменял `k`, потом решил считать nDCG@5 вместо nDCG@10. Каждое «посмотрел» — это
сравнение, и никто их не считает, потому что они разнесены во времени и кажутся
последовательным улучшением, а не перебором.

**Три способа справиться.** Первый — поправка Бонферрони: делить порог на число сравнений.
Просто и слишком консервативно. Второй — контроль доли ложных открытий (Бенджамини-Хохберг):
мягче и практичнее, когда сравнений много. Третий, самый честный и почти никогда не
применяемый: **отложенный набор запросов**, который ты не смотришь, пока не закончил
экспериментировать. Число на нём — единственное, которое можно называть вслух.

**Что это значит для нашего занятия.** Мы сравнили ровно одну пару систем и посчитали `p`
один раз. Это чистый случай. Как только ты начнёшь на неделе 7 подбирать глубину
переранжирования, число сравнений вырастет, и `p = 0,04` перестанет что-либо значить.
Запиши это себе сейчас — в момент подбора об этом не вспоминают никогда.
</details>

⚠️ Ловушка C · **p — не вероятность того, что B лучше.** Это вероятность увидеть такую или
большую разницу, если бы систем**ы не различались вовсе**. Обратное прочтение — самая
распространённая ошибка в отчётах о качестве, и она меняет решения.

⚠️ Ловушка F · **Три теста дали похожие p — это не подтверждение.** Они посчитаны на одних
и тех же пятнадцати числах и потому не независимы. Совпадение p у t-теста, Уилкоксона
и перестановочного означает, что данные не нарушают предпосылок t-теста, а не что эффект
подтверждён трижды.

---

## Задания — 15 мин

Три задания. Первое проверяет реализацию, второе — понимание Гудхарта, третье требует ответа
словами. Каждое заканчивается `assert`-самопроверкой.

**Про самопроверку честно:** пройденная самопроверка не гарантирует, что задание сделано
осмысленно, но проваленная гарантирует, что где-то ошибка.

### Задание 1 · nDCG своими руками против лекции

**Что сделать.** Реализуй `ndcg_at_k(rels, k)` — nDCG, обрезанный по позиции `k`, — и **честно
сравни** результат с числами из `data/l4-metrics.json`. Обрезка означает: и DCG, и IDCG
считаются только по первым `k` позициям.

**Что нужно получить.** `ndcg_full` — nDCG по всей выдаче (`k = 8`), `float`.

**Подсказка.** IDCG@k — это DCG идеального порядка, тоже обрезанного по `k`, а не по всей
выдаче. Перепутать здесь — самая частая ошибка в чужих реализациях.

**Прочитай до запуска.** Исходы:
* совпало с `M["ndcg"]` — верно;
* больше единицы — ты обрезал DCG, но не IDCG;
* меньше ожидаемого на всех `k` — сортируешь по возрастанию, а не по убыванию.

**Формулировка вывода.** Не «моя реализация верна», а: **при каком `k` обрезка перестаёт
влиять** на значение и почему.

In [ ]:
# --- твой код: ЗАДАНИЕ 1 ---
def ndcg_at_k(rels, k):
    ...

ndcg_full = ...
# --- конец ---

assert 0.0 <= ndcg_full <= 1.0, "nDCG обязан лежать в [0,1] -- скорее всего IDCG не обрезан"
assert abs(ndcg_full - M["ndcg"]) < 1e-4, \
    f"не совпало с лекцией: {ndcg_full:.4f} против {M['ndcg']:.4f}"
assert ndcg_at_k(RELS, 1) <= ndcg_at_k(RELS, len(RELS)) or RELS[0] == 1, \
    "при нерелевантном первом документе nDCG@1 обязан быть нулём"
print(f"nDCG@8 = {ndcg_full:.4f} -- совпал с data/l4-metrics.json")

### Задание 2 · Гудхарт: цена оптимизации прокси

**Тезис.** *Порядок, оптимизирующий прокси-метрику, теряет на целевой больше, чем потеряло бы
бездействие.* Проверим буквально: сравним «нагнутый» порядок не только с честным, но и
со **случайным** — с базой из части 2.

В `data/l4-goodhart-steps.json` лежит порядок `gamed`: документы отсортированы по популярности,
а не по соответствию запросу. Это типичная подмена — популярность измерима, дешева и коррелирует
с релевантностью ровно настолько, чтобы казаться разумной.

**Что сделать.** Собрать релевантности в двух порядках и **честно сравнить** три величины:
nDCG нагнутого, nDCG честного и распределение случайного из части 2.

**Что нужно получить.** `gamed_rels`, `honest_rels` — списки релевантностей; `gamed_pct` —
доля случайных перестановок, которые **хуже** нагнутого порядка, `float` в `[0, 1]`.

**Подсказка.** Релевантности уже лежат в `GH["gamed"]["terms"]` и `GH["honest"]["terms"]`,
поле `rel`. Список `samples` со случайными nDCG остался из части 2.

**Прочитай до запуска.** Все исходы содержательны:
* `gamed_pct` мал (меньше 0,1) — оптимизация популярности хуже случайности, тезис подтверждён
  в сильной форме;
* `gamed_pct` около 0,5 — нагнутый порядок неотличим от случайного: прокси не помогает,
  но и не вредит;
* `gamed_pct` велик — популярность на этой коллекции коррелирует с релевантностью, и тогда
  интересен вопрос, почему она всё-таки проигрывает честному порядку.

**Формулировка вывода.** Не «популярность плохой сигнал», а: **при каком условии** ранжирование
по прокси оказывается хуже, чем отсутствие ранжирования вообще.

<details><summary>Как разметить самому, если асессоров нет — рабочая процедура на один вечер</summary>

Асессоров у тебя не будет. Ни на проекте недели 14, ни на первой работе. Размечать придётся
самому, и есть способ сделать это так, чтобы числам можно было верить.

**Шаг 1. Инструкция раньше разметки.** Напиши на полстраницы, что считается релевантным,
с тремя примерами «да» и тремя «нет», включая один спорный. Пиши её **до** того, как увидишь
выдачу: инструкция, написанная по ходу, подстраивается под то, что система уже нашла.

**Шаг 2. Пул, а не выдача одной системы.** Возьми топ-20 от каждой конфигурации, которую
собираешься сравнивать, объедини, перемешай и **сотри признак системы**. Размечая выдачу
по порядку, ты неизбежно щедрее к верхним документам — это то же смещение позиции, что
у пользователей, только теперь оно попадает прямо в разметку.

**Шаг 3. Три градации, не пять.** «Отвечает / частично / мимо». Пять градаций звучат точнее
и дают худшее согласие с самим собой: границу между «4» и «5» никто не держит стабильно.
Если сомневаешься между двумя соседними — ставь меньшую и пометь запрос как спорный.

**Шаг 4. Повторная разметка на подвыборке.** Через день размети заново случайные 15 %
и посчитай каппу **с самим собой**. Если она ниже 0,6, твоя инструкция плоха, и никакие
метрики поверх такой разметки не осмысленны. Это самая дешёвая и самая пропускаемая проверка
из всех.

**Шаг 5. Замерь потолок и назови его.** Сколько документов ты просудил из скольких? Всё
непросуженное станет нулём, и твоя полнота — это полнота относительно пула. Запиши это число
рядом с метриками.

**Чего эта процедура не даёт.** Она не даёт объективности: ты размечаешь под собственное
понимание запроса, и система, обученная на твоём понимании, будет хороша по твоей же разметке.
Единственная защита — заранее написанная инструкция, которую можно показать другому человеку
и спросить, размечал бы он так же.
</details>

<details><summary>Почему nDCG стал стандартом де-факто — и три места, где он всё-таки подводит</summary>

Из всех метрик ранжирования nDCG чаще всего попадает в статьи и лидерборды. У этого есть
причины, и у выбора есть цена.

**Почему он.** Единственная из классических метрик, которая умеет градации релевантности,
учитывает позицию и нормируется в `[0,1]` — то есть даёт число, сравнимое между запросами
и между коллекциями на вид. Плюс он гладко реагирует на перестановки: сдвиг документа
на одну позицию меняет метрику чуть-чуть, а не скачком, как Precision@k. Для сравнения
близких систем это важно.

**Место первое, где подводит: неполная разметка.** nDCG считает непросуженный документ нулевым
выигрышем, то есть таким же, как заведомо нерелевантный. При активном пулинге это занижает
новые системы. bpref и infAP придуманы именно против этого, но в лидерборды не попали.

**Место второе: сравнимость «на вид».** Нормировка делает числа похожими на проценты,
и их начинают сравнивать между коллекциями. одно и то же значение nDCG@10 на MS MARCO и на
BEIR-подмножестве — это не «одинаковое качество», а два числа, посчитанные по разным
разметкам с разными потолками. Между коллекциями сравнимы только **разницы** относительно
общего базового метода, и то с оговорками.

**Место третье: разнообразие и новизна.** nDCG считает документы независимыми. Десять
одинаковых отличных документов дадут ту же метрику, что десять разных отличных, — хотя для
пользователя это провал и успех соответственно. Метрики разнообразия (α-nDCG, ERR-IA)
существуют, считаются сложнее и требуют разметки по подтемам, поэтому применяются редко.

**Вывод не «не используйте nDCG».** Вывод такой: nDCG — хорошая метрика для сравнения систем
на одной коллекции с плотной разметкой. Ровно для этого мы её сегодня и применяли, и ровно
за эти границы её обычно выносят.
</details>

In [ ]:
GH = json.load(open(f"{DATA_DIR}/l4-goodhart-steps.json", encoding="utf-8"))

# --- твой код: ЗАДАНИЕ 2 ---
gamed_rels = ...
honest_rels = ...
gamed_pct = ...
# --- конец ---

assert len(gamed_rels) == len(honest_rels) == len(RELS), "порядки должны быть одной длины"
assert sorted(gamed_rels) == sorted(honest_rels), \
    "это ПЕРЕСТАНОВКИ одного множества -- набор релевантностей обязан совпасть"
assert 0.0 <= gamed_pct <= 1.0, "gamed_pct -- это ДОЛЯ, а не значение nDCG"
g_nd, h_nd = ndcg(gamed_rels), ndcg(honest_rels)
assert g_nd < h_nd, "нагнутый порядок обязан проигрывать честному -- иначе перепутаны местами"
print(f"{'порядок':>12} {'Precision@1':>13} {'nDCG':>8}")
print(f"{'нагнутый':>12} {precision_at_k(gamed_rels, 1):>13.4f} {g_nd:>8.4f}")
print(f"{'честный':>12} {precision_at_k(honest_rels, 1):>13.4f} {h_nd:>8.4f}")
print(f"{'случайный':>12} {'':>13} {BASE:>8.4f}  (среднее по {N_TRIALS} перестановкам)")
print(f"случайных перестановок ХУЖЕ нагнутого: {gamed_pct:.1%}")
RUN["goodhart"] = {"gamed_ndcg": g_nd, "honest_ndcg": h_nd, "gamed_pct": gamed_pct}

### Задание 3 · Словами: сколько запросов нужно

**Что сделать.** В части 4 доверительный интервал получился `[+0,0023; +0,0772]` при пятнадцати
запросах — то есть эффект едва отличим от нуля. Ответь **словами** в ячейке ниже:

1. Во сколько раз надо увеличить число запросов, чтобы половинная ширина интервала уменьшилась
   вдвое, и почему именно так (одна формула, одна фраза объяснения).
2. Назови **один** способ сузить интервал, **не** увеличивая число запросов, и скажи, чем
   за него платят.

**Прочитай до запуска.** `assert` проверяет только объём и что заглушка заменена — содержание
проверяет человек. Ответ, где нет ни числа, ни формулы, проверку пройдёт и ревью не пройдёт.

**Формулировка вывода.** Не «нужно больше данных», а: при каком числе запросов эффект такого
размера стал бы **практически** различимым, и стоит ли он этих запросов.

In [ ]:
# --- твой код: ЗАДАНИЕ 3 ---
ANSWER = """
Впиши ответ сюда: минимум 70 слов, с формулой в первом пункте и ценой во втором.
"""
# --- конец ---

assert len(ANSWER.split()) >= 70, "ответ короче 70 слов -- два пункта так не уместить"
assert "Впиши ответ" not in ANSWER, "заглушка не заменена"
assert any(ch.isdigit() for ch in ANSWER), "в ответе нет ни одного числа -- первый пункт про формулу"
print(f"ответ принят: {len(ANSWER.split())} слов")

---

## Итог занятия — 5 мин

* Построили судейский набор и посчитали, **чего в нём нет**: непросуженное считается
  нерелевантным, и это наказывает находки.
* Посчитали nDCG случайного порядка — и увидели, что наш BM25 на этой коллекции его
  даже не обыгрывает: среднее случайного 0,77 против 0,68 у BM25. Нулевое число —
  не формальность, оно перевернуло вывод занятия.
* Реализовали пять метрик и сверили каждую с числами лекции. Расхождение упало бы ячейкой.
* Показали, что MRR слеп ко всему после первого попадания, а Recall@100 ничего не говорит
  о первом экране.
* Сравнили две системы честно: среднее, знаки по запросам, интервал, три теста — и отказались
  от формулировки «B значимо лучше» в пользу интервала.

**Ограничение нашего замера, которое надо назвать вслух.** Разметка — прокси по категории
20NG, а не суждение человека. Все абсолютные значения метрик поэтому завышены и непереносимы.
Переносимы **отношения и выводы о процедуре**: что интервал важнее p, что среднее скрывает
знаки, что метрика без базы не значит ничего.

**Что мы будем и чего не будем замерять дальше.** На неделе 7 эти же метрики станут мерилом
для каскада — и там впервые появится честное «стало лучше», потому что сравнивать будем
две конфигурации одной системы на одном наборе запросов.

<details><summary>Шесть типов ловушек этого занятия — и почему все они про интерпретацию</summary>

Собери сегодняшние ловушки вместе, и видно, чем это занятие отличается от прошлого. На неделе 3
почти все ловушки были про инструмент и данные: библиотека считает не то, токенизация выбросила
не то. Сегодня центр тяжести сместился.

**A · данных.** Релевантность выведена из категории 20NG — это прокси, а не суждение. Непросуженное
считается нерелевантным, и находка вне пула наказывается.

**B · метрики.** Recall@100 при трёх ссылках на экране. MRR, слепой ко всему после первого
попадания. nDCG, завышенный и нечувствительный на восьми документах.

**C · интерпретации.** Четыре штуки, и это самый населённый тип сегодня: высокая каппа при плохой
разметке; «разница в третьем знаке» как «формулы эквивалентны»; среднее по запросам как
«система лучше»; `p` как вероятность того, что B лучше.

**D · инструмента.** Микро- против макро-усреднения. Перепутанный порядок запросов, ломающий
парность и не выбрасывающий ошибки.

**E · замера.** Сравнение на разных наборах запросов, убивающее парный дизайн и втрое
чувствительность.

**F · переноса.** Три теста с похожими `p` как «подтверждение», хотя они посчитаны на одних
и тех же пятнадцати числах.

**Почему тип C сегодня доминирует.** Потому что метрика — это уже интерпретация. Формула
считается однозначно, а вот что она означает и какое решение из неё следует — не считается
вовсе. Отсюда правило занятия: **арифметика метрики занимает пять минут, а её чтение — всю
карьеру.** Ошибки первого рода видны в код-ревью. Ошибки второго рода попадают в отчёт,
в решение и в продукт.
</details>

<details><summary>Как выглядит честный отчёт об эксперименте — шаблон на пять строк</summary>

Всё занятие было про то, как **не** сказать «B лучше». Вот как сказать правильно. Пять пунктов,
каждый обязателен, порядок имеет значение.

**1. Что сравнивали и на чём.** Две конфигурации, различающиеся ровно одним: чем именно.
Набор запросов: сколько, откуда, как отобраны. Разметка: кем, по какой шкале, какая каппа.
Без этого пункта остальные четыре не читаются.

**2. Метрика и порог, зафиксированные заранее.** «Основная метрика — nDCG@10; катим, если
она выросла и Precision@1 не упала». Слово «заранее» здесь несёт всю нагрузку: метрика,
выбранная после просмотра результатов, — это не метрика, а объяснение.

**3. Эффект с интервалом, а не p.** «+0,040 nDCG@10, 95 % ДИ [0,002; 0,077], n = 15 запросов».
Читатель сам решит, значим ли для него нижний конец интервала. `p` можно привести рядом,
но он не должен стоять первым — иначе он вытеснит интервал из головы читателя.

**4. Где стало хуже.** «На 4 запросах из 15 знак разницы отрицательный; максимальное ухудшение
−0,062 на запросе Q3». Отчёт без этого пункта не рассматривается: усреднение всегда что-то
прячет, и вопрос только в том, покажешь ты это сам или это найдут за тебя.

**5. Чего замер не показывает.** Прямым списком: другое распределение запросов, другая разметка,
онлайн-поведение, стоимость вычислений. Наш семинар заканчивается ровно таким списком, и это
не ритуал — это то, что отличает измерение от рекламы.

**Проверка отчёта одним вопросом.** Если из твоего отчёта нельзя понять, при каких условиях
вывод был бы **противоположным**, отчёт неполон. Утверждение, которое невозможно опровергнуть
никакими данными, ничего и не утверждает.
</details>

In [ ]:
metrics_src = ARTIFACTS / "metrics.py"
metrics_src.write_text(
    "# Реализации, проверенные против data/l4-*.json на семинаре недели 4.\n"
    "import math\n\n"
    "def recall_at_k(rels, k, total_rel):\n"
    "    return sum(rels[:k]) / total_rel if total_rel else 0.0\n\n"
    "def precision_at_k(rels, k):\n"
    "    return sum(rels[:k]) / k if k else 0.0\n\n"
    "def rr(rels):\n"
    "    return next((1 / i for i, r in enumerate(rels, 1) if r), 0.0)\n\n"
    "def ap(rels):\n"
    "    hits, s = 0, 0.0\n"
    "    for i, r in enumerate(rels, 1):\n"
    "        if r:\n"
    "            hits += 1\n"
    "            s += hits / i\n"
    "    return s / hits if hits else 0.0\n\n"
    "def dcg(rels):\n"
    "    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))\n\n"
    "def ndcg(rels):\n"
    "    ideal = dcg(sorted(rels, reverse=True))\n"
    "    return dcg(rels) / ideal if ideal else 0.0\n",
    encoding="utf-8")

print(f"метрики -> {metrics_src} ({metrics_src.stat().st_size} байт)")
print("проверим записанный файл краевыми случаями -- и только потом сохраним замеры")

<details><summary>Ограничения этого семинара, которые надо назвать вслух</summary>

Полный список того, где мы срезали угол, и в какую сторону это смещает выводы.

**Разметка — прокси.** Релевантность выведена из категории 20NG, а не из суждения человека.
Смещение: все абсолютные метрики завышены, потому что внутри категории нерелевантные документы
считаются попаданиями. Направление известно, величина — нет.

**Восемь документов и один запрос в частях 1–3.** Это худший режим для nDCG: потолок близко,
любой порядок неплох, и мы это прямо показали замером случайного порядка. Смещение: разницы
между методами занижены до неразличимости.

**Пятнадцать запросов в части 4 — синтетика.** Числа `systemA`/`systemB` порождены генератором
с фиксированным сидом, а не измерены на реальных системах. Они подобраны так, чтобы эффект
был на границе различимости, — то есть это учебная конструкция, демонстрирующая явление.
Переносить `p = 0,039` куда-либо нельзя.

**Мы не считали онлайн-метрики.** Данные для них есть, но занятие двухчасовое, и мы разобрали
их только в нижнем слое. Это сознательный отказ от замера, и он назван вслух.

**Реализации проверены на краевых случаях — но только на них.** Двенадцать тестов выше
закрывают деление на ноль, вырожденный идеальный порядок и соглашение о Precision@k при `k`
больше длины выдачи. Чего они не проверяют: правильности формул по существу. Проверка
`ndcg([1,1,1]) == 1` пройдёт и у неверной реализации, если она одинаково неверна в числителе
и знаменателе. Настоящую проверку по существу даёт сверка с `data/l4-*.json` в частях 2 и 3 —
там числа приходят из независимого источника.

**Что остаётся долгом.** Дубликаты документов в выдаче: если документ встретился дважды, все
наши метрики посчитают его дважды, и Recall может превысить единицу. В нашей конструкции
дубликатов нет, поэтому и проверки нет, — но на неделе 9, где выдачи двух систем будут
сливаться, они появятся, и там это придётся закрыть.
</details>

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("m", metrics_src)
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)                      # проверяем ЗАПИСАННЫЙ файл, а не то, что в памяти

cases = [
    ("пустая выдача",           lambda: m.ndcg([]),                        0.0),
    ("ни одного релевантного",  lambda: m.ndcg([0, 0, 0]),                 0.0),
    ("все релевантные",         lambda: m.ndcg([1, 1, 1]),                 1.0),
    ("идеальный порядок",       lambda: m.ndcg([1, 1, 0, 0]),              1.0),
    ("худший порядок",          lambda: m.ndcg([0, 0, 1, 1]) < 1.0,        True),
    ("RR без попаданий",        lambda: m.rr([0, 0, 0]),                   0.0),
    ("RR первое место",         lambda: m.rr([1, 0, 0]),                   1.0),
    ("AP без попаданий",        lambda: m.ap([0] * 5),                     0.0),
    ("AP все попадания",        lambda: m.ap([1, 1, 1]),                   1.0),
    ("Precision@0",             lambda: m.precision_at_k([1, 1], 0),       0.0),
    ("Recall при нуле рел.",    lambda: m.recall_at_k([0, 0], 2, 0),       0.0),
    ("Recall не превышает 1",   lambda: m.recall_at_k([1, 1], 2, 2),       1.0),
]
bad = []
for name, fn, want in cases:
    got = fn()
    ok = (got is want or got == want) if isinstance(want, bool) else abs(got - want) < 1e-9
    if not ok:
        bad.append((name, got, want))
assert not bad, f"краевые случаи metrics.py: {bad}"

# Соглашение, которое надо ЗАФИКСИРОВАТЬ, а не оставить на догадку:
# Precision@k при k больше длины выдачи делит на k, а не на длину.
short = m.precision_at_k([1, 1], 10)
assert abs(short - 0.2) < 1e-9, "соглашение изменилось: Precision@k делит на k"
print(f"краевые случаи: {len(cases)}/{len(cases)} пройдены на записанном файле")
print(f"соглашение: Precision@10 от выдачи длины 2 с двумя попаданиями = {short}")
print("           (делим на k -- это метрика страницы фиксированного размера, а не выдачи)")
RUN["edge_cases"] = len(cases)

# Замеры дампим ПОСЛЕ проверок: файл с числами не должен появиться раньше, чем доказано,
# что считавшая их реализация не ломается на краях.
RUN["finished"] = True
(ARTIFACTS / "run-metrics.json").write_text(
    json.dumps(RUN, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"замеры  -> {ARTIFACTS / 'run-metrics.json'} ({len(RUN)} ключей)")
print("на неделе 7 lab-cascade импортирует metrics.py и будет мерить им каскад")

**Что видно.** Двенадцать краевых случаев прошли на **записанном файле**, а не на функциях
в памяти — это разные вещи, и проверять надо второе. Сравнивать надо не сами значения, а
**каждое с тем, что ты считаешь очевидным**: `ndcg([])` равен нулю, а не единице и не NaN,
потому что нулевой IDCG обработан явно; `ndcg([1,1,1])` равен единице, потому что идеальный
порядок совпадает с фактическим. Механизм всех двенадцати одинаков — деление на ноль
и вырожденный идеальный порядок, две ситуации, где разные реализации расходятся молча.
Последняя строка важнее остальных: она **фиксирует соглашение**. Precision@10 от выдачи
длины 2 равен 0,2, а не 1,0, потому что делим на `k`, а не на длину — метрика описывает
страницу фиксированного размера. Чего эта проверка НЕ даёт: гарантии, что формулы верны
по существу; она гарантирует, что они не ломаются на краях и не поменяются молча. Что делать:
дописывать сюда случай каждый раз, когда что-то удивило.

<details><summary>Что понадобится на неделе 7 — и почему метрики уезжают отдельным модулем</summary>

`metrics.py` — это не украшение артефакта. Через три недели на нём будет держаться главное
утверждение семинара про каскад, и вот почему это важно именно так.

**Задача недели 7.** Двухступенчатый каскад: BM25 отбирает сто кандидатов, кросс-энкодер
переранжирует. Вопрос занятия: помогло ли переранжирование и на сколько. Ответ — разница
метрик до и после, посчитанная **той же** реализацией на **тех же** запросах.

**Почему нельзя посчитать заново.** Если на неделе 7 написать nDCG второй раз, любое расхождение
между двумя реализациями замаскируется под эффект переранжирования. Разница между «IDCG обрезан»
и «IDCG не обрезан» на десяти позициях — это сотые, ровно порядок ожидаемого эффекта. Ты будешь
измерять свою же опечатку и назовёшь это выигрышем кросс-энкодера.

**Почему модулем, а не копипастой.** Копия расходится с оригиналом при первой правке, и расходится
молча. Импорт из одного файла гарантирует, что все семинары курса меряют одинаково, а если
реализация окажется неверной, она окажется неверной **одинаково везде** — и это лучше, чем
неверно по-разному, потому что сравнения между занятиями остаются осмысленными.

**Что там будет лежать к концу курса.** К неделе 12 в `artifacts/` соберётся: индекс BM25
(неделя 3), метрики (неделя 4), кэш эмбеддингов и каскад (неделя 7), гибридные скоры (неделя 9),
ANN-индекс (неделя 10), RAG-конвейер (неделя 12). Это не двенадцать разрозненных ноутбуков,
а одна система, собранная по частям, и защищать на неделе 14 ты будешь именно её.

**И тем не менее — каждый семинар запускается один.** Нет `metrics.py` — ноутбук недели 7
определит функции у себя и скажет об этом вслух. Пропуск занятия стоит времени, но не
выкидывает из курса.
</details>

**Что видно.** Модуль на диске, а замеры — ещё нет, и порядок здесь содержательный. Сравнивать
надо не размер файла с чем-либо, а **момент записи с моментом проверки**: файл с числами не должен
появляться раньше, чем доказано, что считавшая их реализация не ломается на краях. Механизм
простой — иначе на диске окажется артефакт, которому нельзя верить, и отличить его от годного
будет нечем. Чего этот шаг НЕ гарантирует: что `metrics.py` корректен по существу — это
показывает сверка с `data/l4-*.json` выше, а не факт записи. Что делать: следующая ячейка
проверяет записанный файл двенадцатью краевыми случаями, и только она сохранит замеры.

---

## Решения

**Подглядеть — не поражение. Поражение — уйти с занятия, не поняв, где был затык.**

<details><summary>Задание 1 · nDCG с обрезкой</summary>

```python
def ndcg_at_k(rels, k):
    d = dcg(rels[:k])
    i = dcg(sorted(rels, reverse=True)[:k])
    return d / i if i else 0.0

ndcg_full = ndcg_at_k(RELS, len(RELS))
```

Ключ в том, что идеальный порядок тоже обрезается по `k`. Если посчитать IDCG по всей выдаче,
а DCG — по первым `k`, значение окажется заниженным; если наоборот — превысит единицу.
Обрезка перестаёт влиять на `k`, начиная с которого в идеальном порядке кончаются релевантные
документы: у нас их четыре, поэтому nDCG@4 и nDCG@8 различаются только за счёт релевантных,
попавших в нашу выдачу позже четвёртого места.
</details>

<details><summary>Задание 2 · Гудхарт</summary>

```python
gamed_rels = [t["rel"] for t in GH["gamed"]["terms"]]
honest_rels = [t["rel"] for t in GH["honest"]["terms"]]
gamed_pct = sum(x < ndcg(gamed_rels) for x in samples) / len(samples)
```

Исход на наших данных сильнее, чем «маленький»: `gamed_pct` равен **нулю** — из двадцати тысяч
случайных перестановок ни одна не оказалась хуже порядка, отсортированного по популярности.
Это предельная форма тезиса: оптимизация прокси не просто не помогла, она дала результат хуже
любого из двадцати тысяч способов не делать ничего.

Условие, при котором так происходит: прокси **антикоррелирует** с целевой величиной на этой
коллекции. Популярные документы здесь — из большой категории `rec.sport.hockey`, а релевантность
определена по `sci.space`. Сортировка по популярности систематически поднимает наверх ровно
то, что размечено нулём, — а случайный порядок хотя бы не имеет систематического уклона.

**Общий вид явления.** Прокси-метрика опасна не тем, что она неточна, а тем, что оптимизация
по ней **направленна**. Шум симметричен и в среднем безвреден; смещение накапливается. Отсюда
практическое правило: прежде чем оптимизировать что-либо, кроме целевой метрики, измерь
корреляцию прокси с целевой — и измерь её на том срезе данных, где будешь оптимизировать,
а не на общем.
</details>

<details><summary>Задание 3 · сколько запросов</summary>

**Первый пункт.** Половинная ширина интервала равна `t·sd/√n`, то есть падает как `1/√n`.
Чтобы уменьшить её вдвое, нужно **вчетверо** больше запросов: с пятнадцати до шестидесяти.
При этом интервал `[0,002; 0,077]` сжался бы примерно до `[0,021; 0,058]` — то есть эффект
стал бы уверенно положительным, но всё ещё небольшим.

**Второй пункт, один из нескольких способов.** Уменьшить `sd` разницы, а не увеличивать `n`:
сравнивать системы на одних и тех же запросах (мы уже так и делаем — это парный дизайн,
и только благодаря ему при пятнадцати запросах вообще что-то видно) либо стратифицировать
запросы по типу и сравнивать внутри страт. Плата: результат становится верным для того
распределения запросов, по которому ты стратифицировал, и хуже переносится на другое.

**Практический вывод.** Эффект в 0,04 nDCG при таком разбросе требует порядка шестидесяти
запросов, чтобы говорить о нём уверенно. Разметка шестидесяти запросов стоит недели работы
асессора — и это ровно то решение, которое надо принимать до эксперимента, а не после.
</details>

---

## Литература

* **Järvelin & Kekäläinen (2002), «Cumulated Gain-Based Evaluation of IR Techniques»** —
  статья, вводящая DCG и nDCG. Читается за час и снимает половину вопросов о том, почему
  дисконт именно логарифмический (ответ: авторы честно пишут, что это выбор, а не вывод).
* **Buckley & Voorhees (2004), «Retrieval Evaluation with Incomplete Information»** — откуда
  взялся bpref и как измеряли смещение от пулинга. Первоисточник к блоку про leave-one-run-out.
* **Moffat & Zobel (2008), «Rank-Biased Precision for Measurement of Retrieval Effectiveness»** —
  RBP с явной моделью пользователя и корректной обработкой неполной разметки.
* **Smucker, Allan & Carterette (2007), «A Comparison of Statistical Significance Tests for IR
  Evaluation»** — почему перестановочный тест предпочтительнее, на реальных TREC-прогонах.
  Прямое основание для части 4.
* **Chapelle et al. (2009), «Expected Reciprocal Rank for Graded Relevance»** — ERR и каскадная
  модель просмотра; полезно рядом с моделью смещения позиции из `data/l4-online.json`.
* **Лекция L5 «Метрики ранжирования»** и `data/l4-*.json` — числа, с которыми мы сверялись.
  Порождены `_research/gen_l4.py` поверх BM25-ранжирования из L3; менять надо генератор, а не JSON.

**Дальше по курсу.** L6 объяснит, чем заменяют совпадение строк, когда упираешься в лексический
потолок с прошлого занятия. L10 даст каскад, а неделя 7 — первый случай, когда «стало лучше»
можно будет сказать честно: одна система, две конфигурации, один набор запросов.

## Дамп прогона

Правило 10.5: занятие не считается прогнанным, пока его числа не лежат в файле рядом
с конфигурацией рантайма. Ячейка ниже собирает все численные результаты ноутбука —
от сида до финальных метрик — и кладёт их в `runs/hw-ranking-metrics.json`. Это и есть
доказательство прогона: разбор сверяется с файлом, а не с памятью автора.

In [ ]:
# Итог прогона (правило 10.5): все числовые результаты + конфигурация рантайма +
# журнал печатей уезжают ОДНИМ архивом. Скачай его по ссылке ниже — и всё.
import json as _json, os as _os, sys as _sys, platform as _pl, shutil as _shutil, base64 as _b64

_runtime = {"python": _sys.version.split()[0], "platform": _pl.platform()}
_torch = _sys.modules.get("torch")   # НЕ импортируем сами: рамка 7.4 — сид и пин
if _torch is not None:                # обязателен только там, где ноутбук torch ИСПОЛЬЗУЕТ
    _runtime["torch"] = _torch.__version__
    _runtime["gpu"] = _torch.cuda.get_device_name(0) if _torch.cuda.is_available() else None
else:
    import subprocess as _sp
    try:
        _q = _sp.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True, timeout=5)
        _runtime["gpu"] = (_q.stdout.strip().splitlines() or [None])[0] if _q.returncode == 0 else None
    except Exception:
        _runtime["gpu"] = None

def _plain(v):
    try:
        import numpy as _np
        if isinstance(v, _np.integer): return int(v)
        if isinstance(v, _np.floating): return float(v)
    except Exception:
        pass
    return v

def _num(v):
    return isinstance(v, (int, float)) and not isinstance(v, bool)

def _tree(v, depth=0):
    """Числовое содержимое величины: само число, либо список/словарь чисел, в том числе
    ВЛОЖЕННЫЙ. Нечисловые ветки отбрасываются, пустое — тоже (None).

    Рекурсия здесь не украшение. Прежний сборщик брал словарь, только если ВСЕ его
    значения — числа, и поэтому целиком терял RUN: там лежит RUN["task1"] = {...},
    один вложенный словарь на весь ноутбук. А RUN — единственное место, где метрика
    записана в тех же единицах, в каких её печатает проза (RUN["bm25_ms"] = t_bm * 1000,
    тогда как в глобалах живут секунды). Из-за этого числа прозы не находились в дампе.
    """
    v = _plain(v)
    if _num(v):
        return v
    if depth >= 4:                     # защита от самоссылающихся структур
        return None
    if isinstance(v, dict) and 0 < len(v) <= 64:
        out = {}
        for _kk, _vv in v.items():
            got = _tree(_vv, depth + 1)
            if got is not None:
                out[str(_kk)] = got
        return out or None
    if isinstance(v, (list, tuple)) and 0 < len(v) <= 64:
        out = [_tree(_x, depth + 1) for _x in v]
        out = [_x for _x in out if _x is not None]
        return out or None
    return None

_metrics = {}
for _k, _v in sorted(globals().items()):
    if _k.startswith("_") or (len(_k) == 1 and _k.islower()):
        continue                       # служебные имена и счётчики циклов
    got = _tree(_v)
    if got is not None:
        _metrics[_k] = got

_payload = {"notebook": NB, "runtime": _runtime, "metrics": _metrics}
_json.dump(_payload, open(RUN_DIR / (NB + ".json"), "w", encoding="utf-8"),
           ensure_ascii=False, indent=1, sort_keys=True)
print(f"величин: {len(_metrics)} · рантайм: {_runtime['gpu'] or 'CPU'}")
_sys.stdout.flush(); _LOG.flush()      # журнал дописан до того, как попадёт в архив

_zip = _shutil.make_archive(str(RUN_DIR), "zip", str(RUN_DIR))
_kb = _os.path.getsize(_zip) / 1024
print(f"архив прогона: {_zip} ({_kb:.0f} КБ)")

# Ссылка на скачивание. Файл живёт в песочнице рантайма и сам до репозитория не доедет,
# а вывод ячейки — доедет: жми ссылку, архив упадёт в загрузки браузера.
_b = _b64.b64encode(open(_zip, "rb").read()).decode()
display(HTML(
    f'<a download="{_os.path.basename(_zip)}" href="data:application/zip;base64,{_b}" '
    f'style="display:inline-block;padding:10px 16px;margin:6px 0;background:#2b4a8b;'
    f'color:#fff;border-radius:4px;text-decoration:none;font-family:sans-serif">'
    f'&#10515; Скачать прогон — {_os.path.basename(_zip)} ({_kb:.0f} КБ)</a>'))
print("скачай архив по ссылке ↑, дальше локально: python3 scripts/import_runs.py")


**Что видно.** В дампе — конфигурация прогона и все скалярные результаты по именам
переменных. Сравнивать надо не тайминги — они свойство рантайма, и на T4, A100 и CPU
законно разные, — а метрики качества: при одном сиде они обязаны совпасть до знака.
Если твой прогон разошёлся с эталонным в качестве, а не во времени, — это находка,
неси её на занятие.